In [ ]:
from MyPlotRecipe.SharedX import ShareXaxis
from MyPlotRecipe.UniversalColor import UniversalColor
from MyPlotRecipe.legend_shadow import legend_shadow
import numpy as np
import math
import datetime

from scipy import signal
from scipy.optimize import curve_fit
from scipy import stats
from scipy.stats import t as t_dist
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
import matplotlib.ticker as ptick
import matplotlib.colors as mplcolors
from matplotlib.ticker import FixedLocator
from matplotlib.ticker import AutoMinorLocator
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.patches import Wedge
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

from CSField import CSField
import JupiterMag as jm

jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

UC = UniversalColor()
UC.set_palette()

F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

In [ ]:
# ============================================================
# 重みづけ最小二乗フィット（1次関数） y = a*x + b
# ============================================================

def weighted_linear_fit(x, y, w):
    """
    重みづけ最小二乗フィット (1次関数)

    Parameters
    ----------
    x     : array-like  説明変数
    y     : array-like  目的変数
    sigma : array-like  各点の標準偏差（測定誤差）

    Returns
    -------
    a, b        : フィット係数 (y = a*x + b)
    sigma_a, sigma_b : 各係数の標準誤差
    """
    # 重みづけ和
    Sw = np.sum(w)
    Swx = np.sum(w * x)
    Swx2 = np.sum(w * x**2)
    Swy = np.sum(w * y)
    Swxy = np.sum(w * x * y)

    # 行列式
    delta = Sw * Swx2 - Swx**2

    # 解析解
    a = (Sw * Swxy - Swx * Swy) / delta
    b = (Swx2 * Swy - Swx * Swxy) / delta

    # 標準誤差
    sigma_a = np.sqrt(Sw / delta)
    sigma_b = np.sqrt(Swx2 / delta)

    return a, b, sigma_a, sigma_b

In [ ]:
def day2hhmm(decimal_day):
    doy = int(decimal_day)
    rest = decimal_day - doy

    hours = int(rest*24)
    minutes = (rest*24-hours)*60

    if round(minutes) >= 60:
        hours += 1
        minutes = 0
    else:
        minutes = round(minutes)
    return hours, minutes


def hhmm2day(hours, minutes, seconds=0):
    fraction_of_day = hours/24 + minutes/(60*24) + seconds/(60*60*24)
    return fraction_of_day


def fmt_hhmm(decimal_day):
    hours, minutes = day2hhmm(decimal_day)
    hh = '{:02}'.format(hours)
    mm = '{:02}'.format(minutes)
    return hh+':'+mm

In [ ]:
# Data from Connerney+2020: PJ index
con20_pj_idx = np.array([1, 3, 4, 5, 6,
                        7, 8, 9, 10, 11,
                        12, 13, 14, 15, 16,
                        17, 18, 19, 20, 21,
                        22, 23, 24], dtype=int)

# Data from Connerney+2020: Current constant [nT]
con20_mu_i_tot = np.array([150.1, 137.8, 127.2, 129.1, 130.1,
                           142.3, 140.1, 143.8, 137.0, 141.4,
                           124.2, 148.9, 145.3, 144.8, 149.9,
                           132.1, 133.5, 152.9, 138.5, 138.8,
                           156.1, 141.4, 146.3])

# Data from Connerney+2020: Radial current constant [MA]
con20_mu_i_rho = np.array([35.2, 14.6, 7.7, 11.5, 20.8,
                           20.2, 12.2, 21.1, 20.9, 10.7,
                           26.26, 16.4, 12.0, 19.6, 12.0,
                           13.6, 20.0, 12.8, 16.0, 17.3,
                           9.9, 16.1, 10.3])

In [ ]:
# 定数
MU0 = 4*np.pi*1E-7       # 真空中の透磁率 [H m-1]
AMU2KG = 1.66E-27        # [kg]
RJ = 71492*1E+3  # Jupiter radius [m]
RE = 1560*1E+3   # Europa radius [m]
# Angular velocity of Jupiter System III rotation [rad sec-1]
OMG_J = 2*np.pi/(9.55*3600)
OMG_G = 2*np.pi/(7.2*24*3600)   # Ganymede orbital angular velosity [rad/sec]

Psyn_eu = (11.22)*3600      # Moon's synodic period [sec]
Psyn_ga = (10.53)*3600      # Moon's synodic period [sec]

In [ ]:
PJ_list = [1, 3, 4, 5,
           6, 7, 8, 9, 10,
           11, 12, 13, 14, 15,
           16, 17, 18, 19, 20,
           21, 22, 23, 24, 25,
           26, 27, 28, 29, 30,
           31, 32, 33, 34, 35,
           36, 37, 38, 39, 40,
           41, 42, 43, 44, 45,
           46, 47, 48, 49, 50,
           51, 52, 53, 54, 55,
           56, 57, 58, 59, 60,
           61, 62, 63, 64, 65,
           66, 67, 68, 69, 70,
           71, 72, 73, 74, 75,
           76, 77, 78, 79, 80]
dir = 'results/insitu_fit_a0_a1_b5_2_v2_36rj'
f0 = np.loadtxt(dir+'/result_PJ' +
                str(PJ_list[0]).zfill(2)+'_PJ'+str(PJ_list[-1]).zfill(2)+'.txt')

In [ ]:
plot_in = 3.0       # Inner edge [RJ]
plot_out = 40.0     # Outer edge [RJ]

In [ ]:
import matplotlib.colors as mplC

colors = ["#005AFF", "#FFFFFF", "#FF3300"]
cmap_bwr = mplC.LinearSegmentedColormap.from_list(
    "red_white_blue", colors, N=256
)

# 全データ

In [ ]:
PJ_num = 6

a_fit_result = []
b_fit_result = []
sig_a_fit_result = []
sig_b_fit_result = []
Juno_MLT_result = []
Juno_pj_result = []
in_out_result = []
RMSE_result = []

for i in range(2):
    PJ_num = PJ_list[i]

    dir = 'data/FGM/PJ'+str(PJ_num).zfill(2)+'/'
    time_arr = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_time.txt')
    Bx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bx.txt')
    By_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_By.txt')
    Bz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bz.txt')
    rx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rx.txt')
    ry_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_ry.txt')
    rz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rz.txt')
    JunoMLT = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_JunoMLT.txt')

    rx_pc *= 1/71492.0  # [RJ]
    ry_pc *= 1/71492.0  # [RJ]
    rz_pc *= 1/71492.0  # [RJ]

    # 座標
    r_pc = np.sqrt(rx_pc**2+ry_pc**2+rz_pc**2)
    cos_phi = rx_pc/r_pc
    sin_phi = ry_pc/r_pc
    cos_theta = rz_pc/r_pc
    sin_theta = np.sqrt(rx_pc**2+ry_pc**2)/r_pc
    # print('Distance [RJ]:', r_pc[0], r_pc[-1])

    # ===========================================================
    # Internal field
    # ===========================================================
    Bx0_pc = np.zeros(time_arr.size)
    By0_pc = np.zeros(time_arr.size)
    Bz0_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        Bx0_pc[i], By0_pc[i], Bz0_pc[i] = jm.Internal.Field(rx_pc[i],
                                                            ry_pc[i],
                                                            rz_pc[i])  # [nT]
    dBx = Bx_pc - Bx0_pc
    dBy = By_pc - By0_pc
    dBz = Bz_pc - Bz0_pc
    dBr = dBx*sin_theta*cos_phi + dBy*sin_theta*sin_phi + dBz*cos_theta
    dBtheta = dBx*cos_theta*cos_phi + dBy*cos_theta*sin_phi - dBz*sin_theta
    dBphi = -dBx*sin_phi + dBy*cos_phi

    # ===========================================================
    # DoE model
    # ===========================================================
    csfield = CSField()

    select_pj = np.where(np.array(PJ_list) == PJ_num)[0][0]
    # print(f0[select_pj, 1], f0[select_pj, 3])

    dBx2_pc = np.zeros(time_arr.size)
    dBy2_pc = np.zeros(time_arr.size)
    dBz2_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        csfield.config(
            i_phi0=f0[select_pj, 1],
            I_rho0=f0[select_pj, 3],
            a0=3.46781481E-6,
            MLT0=0.52113394,
            a1=18.34895438,
            MLT1=20.0143857,
            b1=4.2518,
            MLTb=-0.2086,
        )
        dBx2_pc[i], dBy2_pc[i], dBz2_pc[i] = csfield.magnetic_field_v2(rx_pc[i],
                                                                       ry_pc[i],
                                                                       rz_pc[i],
                                                                       JunoMLT[i])  # [T]

    x_cs, y_cs, z_cs = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
    phi_cs = np.arctan2(y_cs, x_cs)
    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
    dBx2_pc *= 1E+9    # [nT]
    dBy2_pc *= 1E+9    # [nT]
    dBz2_pc *= 1E+9    # [nT]
    dBr2_pc = dBx2_pc*sin_theta*cos_phi + dBy2_pc * \
        sin_theta*sin_phi + dBz2_pc*cos_theta
    dBtheta2_pc = dBx2_pc*cos_theta*cos_phi + \
        dBy2_pc*cos_theta*sin_phi - dBz2_pc*sin_theta
    dBphi2_pc = -dBx2_pc*sin_phi + dBy2_pc*cos_phi

    # ===========================================================
    # RMSE（Root Mean Squared Error）
    # ===========================================================
    x_cs, y_cs, _ = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
    phi_cs = np.arctan2(y_cs, x_cs)
    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
    in_30rj = np.where(r_pc < plot_out)[0][0]
    out_30rj = np.where(r_pc < plot_out)[0][-1]
    if PJ_num in [17]:
        in_5rj = np.where(rho_cs < 8.0)[0][0]
        out_5rj = np.where(rho_cs < 8.0)[0][-1]
    else:
        in_5rj = np.where(rho_cs < plot_in)[0][0]
        out_5rj = np.where(rho_cs < plot_in)[0][-1]
    in_num = in_5rj-in_30rj-1
    out_num = out_30rj-out_5rj-1

    # ===========================================================
    # Sys3 (x, y, z) -> CS (rho, phi, z)
    # ===========================================================
    dBx_cs, dBy_cs, dBz_cs = csfield._sys3_2_cs(dBx, dBy, dBz)
    dBrho_cs = dBx_cs*np.cos(phi_cs) + dBy_cs*np.sin(phi_cs)
    dBphi_cs = -dBx_cs*np.sin(phi_cs) + dBy_cs*np.cos(phi_cs)

    dBx2_cs, dBy2_cs, dBz2_cs = csfield._sys3_2_cs(dBx2_pc, dBy2_pc, dBz2_pc)
    dBrho2_cs = dBx2_cs*np.cos(phi_cs) + dBy2_cs*np.sin(phi_cs)
    dBphi2_cs = -dBx2_cs*np.sin(phi_cs) + dBy2_cs*np.cos(phi_cs)

    # ===========================================================
    # Total RMSE
    # ===========================================================
    RMS_toroidal_cs_inb = math.sqrt(
        (
            np.sum(
                (dBphi_cs[in_30rj:in_5rj]-dBphi2_cs[in_30rj:in_5rj])**2
            )
        )/(in_num)
    )
    RMS_toroidal_cs_outb = math.sqrt(
        (
            np.sum(
                (dBphi_cs[out_5rj:out_30rj]-dBphi2_cs[out_5rj:out_30rj])**2
            )
        )/(out_num)
    )
    print('RMSE (FASTER) [nT]:', round(RMS_toroidal_cs_inb, 2),
          round(RMS_toroidal_cs_outb, 2))

    # ===========================================================
    # Plot boundary
    # ===========================================================
    if PJ_num in [17]:
        r_ticks_ref = np.arange(8.0, plot_out+1, 1)
    else:
        r_ticks_ref = np.arange(plot_in, plot_out+1, 1)
    r_ticks = np.zeros((r_ticks_ref.size*2, 2))
    for i in range(r_ticks_ref.size):
        r_i = r_ticks_ref[-1]-i
        time_i = time_arr[np.where(r_pc < r_i)[0][0]]
        r_ticks[i, 0] = r_i
        r_ticks[i, 1] = time_i

        time_i = time_arr[np.where(r_pc < r_i)[0][-1]]
        r_ticks[-i-1, 0] = r_i
        r_ticks[-i-1, 1] = time_i

    # ===========================================================
    # Plot
    # ===========================================================
    F = ShareXaxis()
    F.fontsize = 22
    F.fontname = 'Liberation Sans Narrow'

    F.set_figparams(nrows=2, figsize=(6.2, 6.5), dpi='L')
    F.initialize()

    F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
                min=-45.0,
                max=45.0,
                ticks=np.arange(-45, 45+1, 9),
                ticklabels=np.abs(np.arange(-45, 45+1, 9)),
                minor_num=3)
    F.set_yaxis(ax_idx=0,
                label=r'$(1/f_z)\cdot \rho_{\rm cs}\cdot \delta B_{\varphi_{\rm cs}}$ [nT$\cdot R_{\rm J}$]',
                min=-300, max=300,
                ticks=np.linspace(-300, 300, 7),
                ticklabels=np.linspace(-300, 300, 7, dtype=int),
                minor_num=10)
    F.set_yaxis(ax_idx=1,
                label=r'MLT [hour]',
                min=0, max=24,
                ticks=np.arange(0, 24, 4),
                ticklabels=np.arange(0, 24, 4, dtype=int),
                minor_num=4)

    # The plot
    inner_idx = np.where((rho_cs < np.min(r_ticks_ref)))[0]
    outer_idx = np.where((r_pc < np.max(r_ticks_ref)))[0]

    rho_cs_inb = rho_cs[in_30rj:in_5rj]
    z_cs_inb = z_cs[in_30rj:in_5rj]
    dBphi_cs_inb = dBphi_cs[in_30rj:in_5rj]
    dBphi2_cs_inb = dBphi2_cs[in_30rj:in_5rj]
    JunoMLT_inb = JunoMLT[in_30rj:in_5rj]
    rho_cs_outb = rho_cs[out_5rj:out_30rj]
    z_cs_outb = z_cs[out_5rj:out_30rj]
    dBphi_cs_outb = dBphi_cs[out_5rj:out_30rj]
    dBphi2_cs_outb = dBphi2_cs[out_5rj:out_30rj]
    JunoMLT_outb = JunoMLT[out_5rj:out_30rj]

    # INBOUND OBSERVATIONS
    g_rho = 0.5*(1.0+np.tanh((rho_cs_inb-3.3)/1.1))
    f_z = -np.sign(z_cs_inb)*g_rho
    f_z = np.where(np.abs(z_cs_inb) > 2.5, f_z, -g_rho*z_cs_inb/2.5)
    eps = 1e-6
    f_z = np.where(np.abs(f_z) < eps, np.sign(f_z)*eps+eps, f_z)
    y_obs = (1/f_z)*rho_cs_inb*dBphi_cs_inb
    F.ax[0].plot(-rho_cs_inb,
                 y_obs,
                 color='k',
                 linewidth=0.7, label='Observations', zorder=2.0)
    F.ax[0].plot(-rho_cs_inb,
                 (1/f_z)*rho_cs_inb*dBphi2_cs_inb,
                 color=UC.red,
                 linewidth=1.8, label='FASTER', zorder=1.0)

    # INBOUND weighted linear fit
    g_rho = 0.5*(1.0+np.tanh((rho_cs_inb-3.3)/1.1))
    f_z = -np.sign(z_cs_inb)*g_rho
    f_z = np.where(np.abs(z_cs_inb) > 2.5, f_z, -g_rho*z_cs_inb/2.5)
    eps = 1e-6
    f_z = np.where(np.abs(f_z) < eps, np.sign(f_z)*eps+eps, f_z)
    a, b, sigma_a, sigma_b = weighted_linear_fit(
        rho_cs_inb*RJ,
        (1/f_z)*(rho_cs_inb*RJ)*(dBphi_cs_inb*1E-9),
        np.abs(f_z))
    a_fit_result += [a]             # beta [T]
    b_fit_result += [b]             # alpha [T m]
    sig_a_fit_result += [sigma_a]   # beta [T]
    sig_b_fit_result += [sigma_b]   # alpha [T m]
    Juno_MLT_result += [JunoMLT_inb[0]]
    in_out_result += [1]
    Juno_pj_result += [PJ_num]
    RMSE_result += [math.sqrt(
        np.sum((y_obs-(a*rho_cs_inb*RJ+b)*1E+9/RJ)**2)/in_num
    )]
    F.ax[0].plot(-rho_cs_inb,
                 (a*rho_cs_inb*RJ+b)*1E+9/RJ,
                 color=UC.blue,
                 linewidth=1.8, label='Weighted', zorder=1.0)

    # OUTBOUND OBSERVATIONS
    g_rho = 0.5*(1.0+np.tanh((rho_cs_outb-3.3)/1.1))
    f_z = -np.sign(z_cs_outb)*g_rho
    f_z = np.where(np.abs(z_cs_outb) > 2.5, f_z, -g_rho*z_cs_outb/2.5)
    eps = 1e-6
    f_z = np.where(np.abs(f_z) < eps, np.sign(f_z)*eps+eps, f_z)
    y_obs = (1/f_z)*rho_cs_outb*dBphi_cs_outb
    F.ax[0].plot(rho_cs_outb,
                 y_obs,
                 color='k',
                 linewidth=0.7, label='Observations', zorder=2.0)
    F.ax[0].plot(rho_cs_outb,
                 (1/f_z)*rho_cs_outb*dBphi2_cs_outb,
                 color=UC.red,
                 linewidth=1.8, label='FASTER', zorder=1.0)

    # OUTBOUND weighted linear fit
    if rho_cs_outb.size > 4:
        g_rho = 0.5*(1.0+np.tanh((rho_cs_outb-3.3)/1.1))
        f_z = -np.sign(z_cs_outb)*g_rho
        f_z = np.where(np.abs(z_cs_outb) > 2.5, f_z, -g_rho*z_cs_outb/2.5)
        eps = 1e-6
        f_z = np.where(np.abs(f_z) < eps, np.sign(f_z)*eps+eps, f_z)
        a, b, sigma_a, sigma_b = weighted_linear_fit(
            rho_cs_outb*RJ,
            (1/f_z)*(rho_cs_outb*RJ)*(dBphi_cs_outb*1E-9),
            np.abs(f_z))
        a_fit_result += [a]             # beta [T]
        b_fit_result += [b]             # alpha [T m]
        sig_a_fit_result += [sigma_a]   # beta [T]
        sig_b_fit_result += [sigma_b]   # alpha [T m]
        Juno_MLT_result += [JunoMLT_outb[-1]]
        in_out_result += [-1]
        Juno_pj_result += [PJ_num]
        RMSE_result += [math.sqrt(
            np.sum((y_obs-(a*rho_cs_outb*RJ+b)*1E+9/RJ)**2)/out_num
        )]
        F.ax[0].plot(rho_cs_outb,
                     (a*rho_cs_outb*RJ+b)*1E+9/RJ,
                     color=UC.blue,
                     linewidth=1.8, label='Weighted', zorder=1.0)

    # MLT
    F.ax[1].plot(-rho_cs_inb,
                 JunoMLT_inb,
                 color='k',
                 linewidth=0.7, label='Observations', zorder=2.0)
    F.ax[1].plot(rho_cs_outb,
                 JunoMLT_outb,
                 color='k',
                 linewidth=0.7, label='Observations', zorder=2.0)
    # Inner shades
    for j in range(2):
        F.ax[j].axvspan(-5.0, 5.0,
                        fc=UC.lightgray, ec=None, zorder=2.5)

    F.ax[0].text(0.09, 0.97, 'PJ'+str(PJ_num).zfill(2),
                 color='k',
                 fontweight='bold',
                 fontsize=F.fontsize,
                 verticalalignment='top',
                 horizontalalignment='left',
                 transform=F.ax[0].transAxes)
    F.ax[0].text(0.02, 0.02, 'Inbound',
                 color='k',
                 fontsize=F.fontsize,
                 verticalalignment='bottom',
                 horizontalalignment='left',
                 transform=F.ax[0].transAxes)
    F.ax[0].text(0.98, 0.02, 'Outbound',
                 color='k',
                 fontsize=F.fontsize,
                 verticalalignment='bottom',
                 horizontalalignment='right',
                 transform=F.ax[0].transAxes)

    plt.savefig('img/B_cs_2components/PJ'+str(PJ_num).zfill(2)+'.png',
                bbox_inches='tight')

    plt.close()

# Field line trace

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=1, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (1800 MLT DUSK)
# FROM NORTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 15.0, 16.0,
              17.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_N = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 北半球から南半球へ
ns = 1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=18.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            result_arr_N[j, 0, k] = x_cs
            result_arr_N[j, 1, k] = y_cs
            result_arr_N[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z > 55.0*RJ:
            print('North edge [RJ]:')
            result_arr_N[j, 0, k:] = np.nan
            result_arr_N[j, 1, k:] = np.nan
            result_arr_N[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_N[j, 0, k:] = np.nan
            result_arr_N[j, 1, k:] = np.nan
            result_arr_N[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if rs < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_N[j, 0, k:] = np.nan
            result_arr_N[j, 1, k:] = np.nan
            result_arr_N[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (1800 MLT DUSK)
# FROM SOUTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([-178.0, -176.0, -174.0, -172.0, -170.0, -168.0, -166.0, -165.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_S = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 南半球から北半球へ
ns = -1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=18.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            result_arr_S[j, 0, k] = x_cs
            result_arr_S[j, 1, k] = y_cs
            result_arr_S[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z < -55.0*RJ:
            print('South edge [RJ]:')
            result_arr_S[j, 0, k:] = np.nan
            result_arr_S[j, 1, k:] = np.nan
            result_arr_S[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_S[j, 0, k:] = np.nan
            result_arr_S[j, 1, k:] = np.nan
            result_arr_S[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if rs < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_S[j, 0, k:] = np.nan
            result_arr_S[j, 1, k:] = np.nan
            result_arr_S[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (0600 MLT DAWN)
# FROM NORTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 15.0, 16.0,
              17.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_N2 = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 北半球から南半球へ
ns = 1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=6.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
            result_arr_N2[j, 0, k] = x_cs
            result_arr_N2[j, 1, k] = y_cs
            result_arr_N2[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z > 55.0*RJ:
            print('North edge [RJ]:')
            result_arr_N2[j, 0, k:] = np.nan
            result_arr_N2[j, 1, k:] = np.nan
            result_arr_N2[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_N2[j, 0, k:] = np.nan
            result_arr_N2[j, 1, k:] = np.nan
            result_arr_N2[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2+z**2) < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_N2[j, 0, k:] = np.nan
            result_arr_N2[j, 1, k:] = np.nan
            result_arr_N2[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (0600 MLT DAWN)
# FROM SOUTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([-178.0, -176.0, -174.0, -172.0, -170.0, -168.0, -166.0, -165.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_S2 = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 南半球から北半球へ
ns = -1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=6.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
            result_arr_S2[j, 0, k] = x_cs
            result_arr_S2[j, 1, k] = y_cs
            result_arr_S2[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z < -55.0*RJ:
            print('South edge [RJ]:')
            result_arr_S2[j, 0, k:] = np.nan
            result_arr_S2[j, 1, k:] = np.nan
            result_arr_S2[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_S2[j, 0, k:] = np.nan
            result_arr_S2[j, 1, k:] = np.nan
            result_arr_S2[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2+z**2) < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_S2[j, 0, k:] = np.nan
            result_arr_S2[j, 1, k:] = np.nan
            result_arr_S2[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (0300 MLT DAWN)
# FROM NORTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 15.0, 16.0,
              17.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_N3 = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 北半球から南半球へ
ns = 1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=6.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
            result_arr_N3[j, 0, k] = x_cs
            result_arr_N3[j, 1, k] = y_cs
            result_arr_N3[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z > 55.0*RJ:
            print('North edge [RJ]:')
            result_arr_N3[j, 0, k:] = np.nan
            result_arr_N3[j, 1, k:] = np.nan
            result_arr_N3[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_N3[j, 0, k:] = np.nan
            result_arr_N3[j, 1, k:] = np.nan
            result_arr_N3[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2+z**2) < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_N3[j, 0, k:] = np.nan
            result_arr_N3[j, 1, k:] = np.nan
            result_arr_N3[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ==============================================================
# FIELD LINE TRACE IN THE CURRENT SHEET PLANE (0300 MLT DAWN)
# FROM SOUTHERN IONOSPHERE
# ==============================================================
r_cs = (66854.0+500.0)*1E+3  # [m]
theta_cs = np.radians(
    np.array([-178.0, -176.0, -174.0, -172.0, -170.0, -168.0, -166.0, -165.0])
)  # Colatitude [rad]
phi_cs = 0.0  # [rad]

R = csfield.R_sys3_2_cs()
RT = csfield.R_cs_2_sys3()

x_cs0 = r_cs*np.sin(theta_cs)*np.cos(phi_cs)
y_cs0 = r_cs*np.sin(theta_cs)*np.sin(phi_cs)
z_cs0 = r_cs*np.cos(theta_cs)

Niter = 50000

ds0 = 500000.0   # [m]
p = 1.2

result_arr_S3 = np.zeros((x_cs0.size, 3, int(Niter/10)))

# 南半球から北半球へ
ns = -1     # -1: north; 1: south
for j in range(x_cs0.size):
    k = 0
    pos_s3_0 = RT @ np.array([
        x_cs0[j], y_cs0[j], z_cs0[j]
    ])                      # [m]
    x = pos_s3_0[0].copy()  # [m]
    y = pos_s3_0[1].copy()  # [m]
    z = pos_s3_0[2].copy()  # [m]
    for i in range(Niter):
        Bx0, By0, Bz0 = jm.Internal.Field(x/RJ, y/RJ, z/RJ)       # [nT]
        Bx1, By1, Bz1 = csfield.magnetic_field_v2(x/RJ, y/RJ, z/RJ,
                                                  MLT=6.0)  # [T]
        Bx = (Bx0*1E-9)+Bx1     # [T]
        By = (By0*1E-9)+By1     # [T]
        Bz = (Bz0*1E-9)+Bz1     # [T]

        # Sys III -> CS
        B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
        rho_cs = np.sqrt(x_cs**2 + y_cs**2)
        phi_cs = np.arctan2(y_cs, x_cs)
        B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
        B_phi_cs = 0.0     # rho_cs - z_cs plane
        B_z_cs = B_vec_cs[2]
        B_vec_cs = np.array([
            B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
            B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
            B_z_cs
        ])

        # CS -> Sys III
        B_vec_s3 = RT @ B_vec_cs
        B0 = math.sqrt(
            B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

        ds = ds0/(np.log10(B0*1E+9))**p      # [m]

        # 座標更新 (x, y, z)
        x += (ds*B_vec_s3[0]/B0)*ns
        y += (ds*B_vec_s3[1]/B0)*ns
        z += (ds*B_vec_s3[2]/B0)*ns

        # 座標更新 (r, theta, phi)
        rs = math.sqrt(x**2 + y**2 + z**2)
        theta = math.acos(z/rs)
        phi = math.atan2(y, x)

        if i % 10 == 0:
            x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
            result_arr_S3[j, 0, k] = x_cs
            result_arr_S3[j, 1, k] = y_cs
            result_arr_S3[j, 2, k] = z_cs
            k += 1

        # 終了条件 (北の外側境界)
        if z < -55.0*RJ:
            print('South edge [RJ]:')
            result_arr_S3[j, 0, k:] = np.nan
            result_arr_S3[j, 1, k:] = np.nan
            result_arr_S3[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2) > 60.0*RJ:
            print('Horizontal edge')
            result_arr_S3[j, 0, k:] = np.nan
            result_arr_S3[j, 1, k:] = np.nan
            result_arr_S3[j, 2, k:] = np.nan
            break
        # 終了条件 (水平距離の外側境界)
        if math.sqrt(x**2+y**2+z**2) < (66854.0+500.0-1.0)*1E+3:
            print('Jupiter surface')
            result_arr_S3[j, 0, k:] = np.nan
            result_arr_S3[j, 1, k:] = np.nan
            result_arr_S3[j, 2, k:] = np.nan
            break

    print(rs/RJ, y/RJ)

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor((0.2, 0.2, 0.2))
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    def draw_Galilean_moons(ax, facealpha=0.0):
        ax.axhline(y=0.0, color=UC.gray, linewidth=0.35, zorder=7)
        radii = [5.9, 9.4, 15.0, 26.0]
        name = ['Io', 'Eu', 'Ga', 'Ca']
        labelz = [-1.5, -1.5, -1.5, -1.5]
        for i in range(len(radii)):
            ax.add_patch(plt.Circle((radii[i], 0), 0.19, ec='k', fc=UC.gray,
                                    zorder=8, fill=True, clip_on=True))
            F.textbox(ax_idx=ax,
                      x=radii[i], y=labelz[i],
                      text=name[i],
                      textedge=True,
                      fontsize=F.fontsize*0.65,
                      facealpha=facealpha,
                      edgecolor='none',
                      horizontalalignment='center',
                      zorder=8)
        return None
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    def hatch_outer_36rj(ax):
        for r_in, r_out in [(36, 41)]:
            # 上半分（z_cs > 0, つまり角度0-180度）
            wedge_top = Wedge(
                center=(0, 0),      # 木星の位置
                r=r_out,            # 外半径
                theta1=0,           # 開始角度
                theta2=360,         # 終了角度
                width=r_out - r_in,  # 幅 = 4 RJ
                facecolor='k',
                edgecolor=None,
                alpha=0.30,
                linewidth=3.0,
                transform=ax.transData,
                zorder=4.9,
            )
            ax.add_patch(wedge_top)
            ax.axvspan(0, 5, fc='k', ec='none', zorder=4.9, alpha=1.0)
        return None
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Magnetic field line
# ===========================================================
def draw_field_line_0300(ax):
    for i in range(result_arr_N3.shape[0]):
        rho_cs = np.sqrt(result_arr_N3[i, 0, :]**2+result_arr_N3[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_N3[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    for i in range(result_arr_S3.shape[0]):
        rho_cs = np.sqrt(result_arr_S3[i, 0, :]**2+result_arr_S3[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_S3[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    return None


def draw_field_line_0600(ax):
    for i in range(result_arr_N2.shape[0]):
        rho_cs = np.sqrt(result_arr_N2[i, 0, :]**2+result_arr_N2[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_N2[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    for i in range(result_arr_S2.shape[0]):
        rho_cs = np.sqrt(result_arr_S2[i, 0, :]**2+result_arr_S2[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_S2[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    return None


draw_field_line_0300(ax)

# for i in range(result_arr_S2.shape[0]):
#     rho_cs = np.sqrt(result_arr_S2[i, 0, :]**2+result_arr_S2[i, 1, :]**2)
#     ax.plot(rho_cs, -result_arr_S2[i, 2, :],
#             linewidth=0.55, color=UC.yellow, zorder=5)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================


def dawn_midnight_latitude_label_left(ax):
    r_arc = 41.1  # RJ

    lat_deg_list = [-64.5, -54, -43, -31, -17.5, -10, -1.8, 10, 17]
    label_list = [6, 8, 10, 12, 14, 15, 16, 15, 14]
    for i in range(len(lat_deg_list)):
        lat_deg = lat_deg_list[i]
        lat_rad = np.deg2rad(lat_deg)
        rho_i = r_arc * np.cos(lat_rad)
        z_i = r_arc * np.sin(lat_rad)

        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'

        ax.text(
            rho_i, z_i, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [1, 3.2, 5, 7.5, 11, 18]
    label_list = [2, 4, 6, 8, 10, 12]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], 14, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [4, 11]
    label_list = [2, 4]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], -39.8, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='bottom',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [9, 13, 18]
    label_list = [20, 18, 17]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], 2, label,
            fontsize=F.fontsize*0.45,
            color='k',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    return None


dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Magnetic field line
# ===========================================================
def draw_field_line_1800(ax):
    for i in range(result_arr_N.shape[0]):
        rho_cs = np.sqrt(result_arr_N[i, 0, :]**2+result_arr_N[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_N[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    for i in range(result_arr_S.shape[0]):
        rho_cs = np.sqrt(result_arr_S[i, 0, :]**2+result_arr_S[i, 1, :]**2)
        ax.plot(rho_cs, result_arr_S[i, 2, :],
                linewidth=0.55, color=UC.gray, zorder=5)
    return None


draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
def dusk_latitude_label_right(ax):
    r_arc = 41.1  # RJ

    lat_deg_list = [-63.5, -53, -42, -30, -16.5, -9.5, 9, 16.5]
    label_list = [6, 8, 10, 12, 14, 15, 15, 14]
    for i in range(len(lat_deg_list)):
        lat_deg = lat_deg_list[i]
        lat_rad = np.deg2rad(lat_deg)
        rho_i = r_arc * np.cos(lat_rad)
        z_i = r_arc * np.sin(lat_rad)

        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'

        ax.text(
            rho_i, z_i, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [1.5, 3.2, 5.5, 8.0, 12, 19]
    label_list = [2, 4, 6, 8, 10, 12]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], 14, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [4.5, 11.5]
    label_list = [2, 4]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], -39.8, label,
            fontsize=F.fontsize*0.45,
            color='white',
            ha='center', va='bottom',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [9, 13, 17]
    label_list = [20, 18, 17]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], 2, label,
            fontsize=F.fontsize*0.45,
            color='k',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    lat_x_list = [33.5]
    label_list = [16]
    for i in range(len(lat_x_list)):
        # ラベル文字列（例: "+10°", "-8°", "0°"）
        label = f'{label_list[i]}°' if lat_deg != 0 else '0°'
        ax.text(
            lat_x_list[i], -2.5, label,
            fontsize=F.fontsize*0.45,
            color='k',
            ha='center', va='center',
            rotation=0,
            zorder=10,
        )
    return None


dusk_latitude_label_right(ax)

fig.suptitle('Magnetic field morphology', y=0.95,
             fontsize=F.fontsize, fontweight="bold")

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor((0.2, 0.2, 0.2))
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0300(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)

fig.suptitle('Magnetic field morphology', y=0.95,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/field_lines_schematic.svg',
            bbox_inches='tight')

# Selected PJ

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

In [ ]:
PJ_num = 5

dir = 'data/FGM/PJ'+str(PJ_num).zfill(2)+'/'
time_arr = np.loadtxt(
    dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_time.txt')
Bx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bx.txt')
By_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_By.txt')
Bz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bz.txt')
rx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rx.txt')
ry_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_ry.txt')
rz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rz.txt')
JunoMLT = np.loadtxt(
    dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_JunoMLT.txt')

rx_pc *= 1/71492.0  # [RJ]
ry_pc *= 1/71492.0  # [RJ]
rz_pc *= 1/71492.0  # [RJ]

# 座標
r_pc = np.sqrt(rx_pc**2+ry_pc**2+rz_pc**2)
cos_phi = rx_pc/r_pc
sin_phi = ry_pc/r_pc
cos_theta = rz_pc/r_pc
sin_theta = np.sqrt(rx_pc**2+ry_pc**2)/r_pc
# print('Distance [RJ]:', r_pc[0], r_pc[-1])

# ===========================================================
# Internal field
# ===========================================================
Bx0_pc = np.zeros(time_arr.size)
By0_pc = np.zeros(time_arr.size)
Bz0_pc = np.zeros(time_arr.size)
for i in range(time_arr.size):
    Bx0_pc[i], By0_pc[i], Bz0_pc[i] = jm.Internal.Field(rx_pc[i],
                                                        ry_pc[i],
                                                        rz_pc[i])  # [nT]
dBx = Bx_pc - Bx0_pc
dBy = By_pc - By0_pc
dBz = Bz_pc - Bz0_pc
dBr = dBx*sin_theta*cos_phi + dBy*sin_theta*sin_phi + dBz*cos_theta
dBtheta = dBx*cos_theta*cos_phi + dBy*cos_theta*sin_phi - dBz*sin_theta
dBphi = -dBx*sin_phi + dBy*cos_phi

# ===========================================================
# DoE model
# ===========================================================
csfield = CSField()

select_pj = np.where(np.array(PJ_list) == PJ_num)[0][0]
# print(f0[select_pj, 1], f0[select_pj, 3])

dBx2_pc = np.zeros(time_arr.size)
dBy2_pc = np.zeros(time_arr.size)
dBz2_pc = np.zeros(time_arr.size)
for i in range(time_arr.size):
    csfield.config(
        i_phi0=f0[select_pj, 1],
        I_rho0=f0[select_pj, 3],
        a0=3.46781481E-6,
        MLT0=0.52113394,
        a1=18.34895438,
        MLT1=20.0143857,
        b1=4.2518,
        MLTb=-0.2086,
    )
    dBx2_pc[i], dBy2_pc[i], dBz2_pc[i] = csfield.magnetic_field_v2(rx_pc[i],
                                                                   ry_pc[i],
                                                                   rz_pc[i],
                                                                   JunoMLT[i])  # [T]

x_cs, y_cs, z_cs = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
phi_cs = np.arctan2(y_cs, x_cs)
rho_cs = np.sqrt(x_cs**2 + y_cs**2)
dBx2_pc *= 1E+9    # [nT]
dBy2_pc *= 1E+9    # [nT]
dBz2_pc *= 1E+9    # [nT]
dBr2_pc = dBx2_pc*sin_theta*cos_phi + dBy2_pc * \
    sin_theta*sin_phi + dBz2_pc*cos_theta
dBtheta2_pc = dBx2_pc*cos_theta*cos_phi + \
    dBy2_pc*cos_theta*sin_phi - dBz2_pc*sin_theta
dBphi2_pc = -dBx2_pc*sin_phi + dBy2_pc*cos_phi

# ===========================================================
# Inner & Outer Edges
# ===========================================================
x_cs, y_cs, _ = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
phi_cs = np.arctan2(y_cs, x_cs)
rho_cs = np.sqrt(x_cs**2 + y_cs**2)
in_30rj = np.where(r_pc < plot_out)[0][0]
out_30rj = np.where(r_pc < plot_out)[0][-1]
if PJ_num in [17]:
    in_5rj = np.where(rho_cs < 8.0)[0][0]
    out_5rj = np.where(rho_cs < 8.0)[0][-1]
else:
    in_5rj = np.where(rho_cs < plot_in)[0][0]
    out_5rj = np.where(rho_cs < plot_in)[0][-1]
in_num = in_5rj-in_30rj-1
out_num = out_30rj-out_5rj-1

# ===========================================================
# Sys3 (x, y, z) -> CS (rho, phi, z)
# ===========================================================
dBx_cs, dBy_cs, dBz_cs = csfield._sys3_2_cs(dBx, dBy, dBz)
dBrho_cs = dBx_cs*np.cos(phi_cs) + dBy_cs*np.sin(phi_cs)
dBphi_cs = -dBx_cs*np.sin(phi_cs) + dBy_cs*np.cos(phi_cs)

dBx2_cs, dBy2_cs, dBz2_cs = csfield._sys3_2_cs(dBx2_pc, dBy2_pc, dBz2_pc)
dBrho2_cs = dBx2_cs*np.cos(phi_cs) + dBy2_cs*np.sin(phi_cs)
dBphi2_cs = -dBx2_cs*np.sin(phi_cs) + dBy2_cs*np.cos(phi_cs)

# ===========================================================
# Total RMSE
# ===========================================================
RMS_toroidal_cs_inb = math.sqrt(
    (
        np.sum(
            (dBphi_cs[in_30rj:in_5rj]-dBphi2_cs[in_30rj:in_5rj])**2
        )
    )/(in_num)
)
RMS_toroidal_cs_outb = math.sqrt(
    (
        np.sum(
            (dBphi_cs[out_5rj:out_30rj]-dBphi2_cs[out_5rj:out_30rj])**2
        )
    )/(out_num)
)
print('RMSE (FASTER) [nT]:', round(RMS_toroidal_cs_inb, 2),
      round(RMS_toroidal_cs_outb, 2))

# ===========================================================
# Plot boundary
# ===========================================================
if PJ_num in [17]:
    r_ticks_ref = np.arange(8.0, plot_out+1, 1)
else:
    r_ticks_ref = np.arange(plot_in, plot_out+1, 1)
r_ticks = np.zeros((r_ticks_ref.size*2, 2))
for i in range(r_ticks_ref.size):
    r_i = r_ticks_ref[-1]-i
    time_i = time_arr[np.where(r_pc < r_i)[0][0]]
    r_ticks[i, 0] = r_i
    r_ticks[i, 1] = time_i

    time_i = time_arr[np.where(r_pc < r_i)[0][-1]]
    r_ticks[-i-1, 0] = r_i
    r_ticks[-i-1, 1] = time_i

# ===========================================================
# B_phi residuals
# ===========================================================
Bphi_cs_res = dBphi_cs-dBphi2_cs

In [ ]:
# ===========================================================
# Plot
# ===========================================================
F = ShareXaxis()
F.fontsize = 22
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.2, 5.), dpi='XL')
F.initialize()

F.ax.set_aspect('equal')

F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
            min=0.0, max=30.0,
            ticks=np.arange(0, 30+1, 5),
            ticklabels=np.arange(0, 30+1, 5),
            minor_num=5)
F.set_yaxis(ax_idx=0,
            label=r'$z_{\rm cs}$ [$R_{\rm J}$]',
            min=-15.0, max=15.0,
            ticks=np.arange(-15, 15+1, 5),
            ticklabels=np.arange(-15, 15+1, 5, dtype=int),
            minor_num=5,)

cmap = plt.get_cmap('RdYlBu')
F.ax.scatter(rho_cs[in_30rj:in_5rj],
             z_cs[in_30rj:in_5rj],
             c=Bphi_cs_res[in_30rj:in_5rj],
             vmin=-20, vmax=20, cmap=cmap,
             s=10)
F.ax.scatter(rho_cs[out_5rj:out_30rj],
             z_cs[out_5rj:out_30rj],
             c=Bphi_cs_res[out_5rj:out_30rj],
             vmin=-20, vmax=20, cmap=cmap,
             s=10)

# z_cs current sheet shade
rng = np.random.default_rng(40)
noise = rng.random((500, 1500))
weight = np.clip(np.linspace(0.0, 50.0, noise.shape[1])/7.5, 0.0, 1.0)**2.0
noise *= weight[None, :]
im = F.ax.imshow(
    noise,
    extent=[0.0, 50.0, -2.5, 2.5],
    origin="lower",
    cmap=F.IDL_blue_white(reverse=True),
    alpha=0.28,
    interpolation="nearest",
    aspect="auto",
)

# Jupiter
F.ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                          zorder=2, fill=True, clip_on=True))

# Galilean moons
F.ax.axhline(y=0.0, color='k', linewidth=0.2)
radii = [5.9, 9.4, 15.0, 26.0]
name = ['Io', 'Eu', 'Ga', 'Ca']
labelz = [-0.4, -0.4, -0.4, -0.4]
for i in range(len(radii)):
    F.ax.add_patch(plt.Circle((radii[i], 0), 0.15, color='k',
                              zorder=2, fill=True, clip_on=True))
    F.ax.text(radii[i]+0.05, labelz[i], name[i],
              color='k',
              fontweight='bold',
              fontsize=F.fontsize*0.45,
              verticalalignment='top',
              horizontalalignment='left',)

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

# ===========================================================
# Initialize the list
# ===========================================================
Brho_cs_res_5_36 = []
Bphi_cs_res_5_36 = []
Bz_cs_res_5_36 = []
dBrho_cs_5_36 = []
dBphi_cs_5_36 = []
dBz_cs_5_36 = []
rho_cs_5_36 = []
phi_cs_5_36 = []
z_cs_5_36 = []
JunoMLT_5_36 = []
JunoPJ_5_36 = []

# ===========================================================
# Plot settings
# ===========================================================
F = ShareXaxis()
F.fontsize = 17
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(6.2, 5.5), dpi='XL')
F.initialize()

F.ax.set_aspect('equal')

F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
            min=0.0, max=40.0,
            ticks=np.arange(0, 45+1, 5),
            ticklabels=np.arange(0, 45+1, 5),
            minor_num=5)
F.set_yaxis(ax_idx=0,
            label=r'$z_{\rm cs}$ [$R_{\rm J}$]',
            min=-40.0, max=15.0,
            ticks=np.arange(-40, 15+1, 5),
            ticklabels=np.arange(-40, 15+1, 5, dtype=int),
            minor_num=5,)
F.ax.set_facecolor('k')
cmap = plt.get_cmap('RdBu_r')


# ===========================================================
# Data load and plot the B_phi residuals
# ===========================================================
for i in range(len(PJ_list)):
    PJ_num = PJ_list[i]

    dir = 'data/FGM/PJ'+str(PJ_num).zfill(2)+'/'
    time_arr = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_time.txt')
    Bx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bx.txt')
    By_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_By.txt')
    Bz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bz.txt')
    rx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rx.txt')
    ry_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_ry.txt')
    rz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rz.txt')
    JunoMLT = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_JunoMLT.txt')

    rx_pc *= 1/71492.0  # [RJ]
    ry_pc *= 1/71492.0  # [RJ]
    rz_pc *= 1/71492.0  # [RJ]

    # 座標
    r_pc = np.sqrt(rx_pc**2+ry_pc**2+rz_pc**2)
    cos_phi = rx_pc/r_pc
    sin_phi = ry_pc/r_pc
    cos_theta = rz_pc/r_pc
    sin_theta = np.sqrt(rx_pc**2+ry_pc**2)/r_pc
    # print('Distance [RJ]:', r_pc[0], r_pc[-1])

    # ===========================================================
    # Internal field
    # ===========================================================
    Bx0_pc = np.zeros(time_arr.size)
    By0_pc = np.zeros(time_arr.size)
    Bz0_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        Bx0_pc[i], By0_pc[i], Bz0_pc[i] = jm.Internal.Field(rx_pc[i],
                                                            ry_pc[i],
                                                            rz_pc[i])  # [nT]
    dBx = Bx_pc - Bx0_pc
    dBy = By_pc - By0_pc
    dBz = Bz_pc - Bz0_pc
    dBr = dBx*sin_theta*cos_phi + dBy*sin_theta*sin_phi + dBz*cos_theta
    dBtheta = dBx*cos_theta*cos_phi + dBy*cos_theta*sin_phi - dBz*sin_theta
    dBphi = -dBx*sin_phi + dBy*cos_phi

    # ===========================================================
    # DoE model
    # ===========================================================
    csfield = CSField()

    select_pj = np.where(np.array(PJ_list) == PJ_num)[0][0]
    # print(f0[select_pj, 1], f0[select_pj, 3])

    dBx2_pc = np.zeros(time_arr.size)
    dBy2_pc = np.zeros(time_arr.size)
    dBz2_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        csfield.config(
            i_phi0=f0[select_pj, 1],
            I_rho0=f0[select_pj, 3],
            a0=3.46781481E-6,
            MLT0=0.52113394,
            a1=18.34895438,
            MLT1=20.0143857,
            b1=4.2518,
            MLTb=-0.2086,
        )
        dBx2_pc[i], dBy2_pc[i], dBz2_pc[i] = csfield.magnetic_field_v2(rx_pc[i],
                                                                       ry_pc[i],
                                                                       rz_pc[i],
                                                                       JunoMLT[i])  # [T]

    x_cs, y_cs, z_cs = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
    phi_cs = np.arctan2(y_cs, x_cs)
    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
    dBx2_pc *= 1E+9    # [nT]
    dBy2_pc *= 1E+9    # [nT]
    dBz2_pc *= 1E+9    # [nT]
    dBr2_pc = dBx2_pc*sin_theta*cos_phi + dBy2_pc * \
        sin_theta*sin_phi + dBz2_pc*cos_theta
    dBtheta2_pc = dBx2_pc*cos_theta*cos_phi + \
        dBy2_pc*cos_theta*sin_phi - dBz2_pc*sin_theta
    dBphi2_pc = -dBx2_pc*sin_phi + dBy2_pc*cos_phi

    # ===========================================================
    # Inner & Outer Edges
    # ===========================================================
    x_cs, y_cs, _ = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
    phi_cs = np.arctan2(y_cs, x_cs)
    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
    in_30rj = np.where(r_pc < plot_out)[0][0]
    out_30rj = np.where(r_pc < plot_out)[0][-1]
    if PJ_num in [17]:
        in_5rj = np.where(rho_cs < 8.0)[0][0]
        out_5rj = np.where(rho_cs < 8.0)[0][-1]
    else:
        in_5rj = np.where(rho_cs < plot_in)[0][0]
        out_5rj = np.where(rho_cs < plot_in)[0][-1]
    in_num = in_5rj-in_30rj-1
    out_num = out_30rj-out_5rj-1

    # ===========================================================
    # Sys3 (x, y, z) -> CS (rho, phi, z)
    # ===========================================================
    dBx_cs, dBy_cs, dBz_cs = csfield._sys3_2_cs(dBx, dBy, dBz)
    dBrho_cs = dBx_cs*np.cos(phi_cs) + dBy_cs*np.sin(phi_cs)
    dBphi_cs = -dBx_cs*np.sin(phi_cs) + dBy_cs*np.cos(phi_cs)

    dBx2_cs, dBy2_cs, dBz2_cs = csfield._sys3_2_cs(dBx2_pc, dBy2_pc, dBz2_pc)
    dBrho2_cs = dBx2_cs*np.cos(phi_cs) + dBy2_cs*np.sin(phi_cs)
    dBphi2_cs = -dBx2_cs*np.sin(phi_cs) + dBy2_cs*np.cos(phi_cs)

    # ===========================================================
    # B_rho_cs residuals
    # ===========================================================
    Brho_cs_res = dBrho_cs-dBrho2_cs

    # ===========================================================
    # B_phi_cs residuals
    # ===========================================================
    Bphi_cs_res = dBphi_cs-dBphi2_cs

    # ===========================================================
    # B_z_cs residuals
    # ===========================================================
    Bz_cs_res = dBz_cs-dBz2_cs

    # ===========================================================
    # Color coded plot
    # ===========================================================
    F.ax.scatter(rho_cs[in_30rj:in_5rj],
                 z_cs[in_30rj:in_5rj],
                 c=Bphi_cs_res[in_30rj:in_5rj],
                 vmin=-20, vmax=20, cmap=cmap,
                 s=0.2,
                 rasterized=True,)
    F.ax.scatter(rho_cs[out_5rj:out_30rj],
                 z_cs[out_5rj:out_30rj],
                 c=Bphi_cs_res[out_5rj:out_30rj],
                 vmin=-20, vmax=20, cmap=cmap,
                 s=0.2,
                 rasterized=True,)

    # ===========================================================
    # Save to the list
    # ===========================================================
    Brho_cs_res_5_36 += Brho_cs_res[in_30rj:in_5rj].tolist()
    Bphi_cs_res_5_36 += Bphi_cs_res[in_30rj:in_5rj].tolist()
    Bz_cs_res_5_36 += Bz_cs_res[in_30rj:in_5rj].tolist()
    rho_cs_5_36 += rho_cs[in_30rj:in_5rj].tolist()
    phi_cs_5_36 += phi_cs[in_30rj:in_5rj].tolist()
    z_cs_5_36 += z_cs[in_30rj:in_5rj].tolist()
    JunoMLT_5_36 += JunoMLT[in_30rj:in_5rj].tolist()
    # JunoPJ_5_36 += (PJ_num*np.ones(in_5rj-in_30rj+1)).tolist()
    JunoPJ_5_36 += (PJ_num*np.ones(in_5rj-in_30rj)).tolist()

    Brho_cs_res_5_36 += Brho_cs_res[out_5rj:out_30rj].tolist()
    Bphi_cs_res_5_36 += Bphi_cs_res[out_5rj:out_30rj].tolist()
    Bz_cs_res_5_36 += Bz_cs_res[out_5rj:out_30rj].tolist()
    rho_cs_5_36 += rho_cs[out_5rj:out_30rj].tolist()
    phi_cs_5_36 += phi_cs[out_5rj:out_30rj].tolist()
    z_cs_5_36 += z_cs[out_5rj:out_30rj].tolist()
    JunoMLT_5_36 += JunoMLT[out_5rj:out_30rj].tolist()
    # JunoPJ_5_36 += (PJ_num*np.ones(out_30rj-out_5rj+1)).tolist()
    JunoPJ_5_36 += (PJ_num*np.ones(out_30rj-out_5rj)).tolist()

    dBrho_cs_5_36 += dBrho_cs[in_30rj:in_5rj].tolist()
    dBrho_cs_5_36 += dBrho_cs[out_5rj:out_30rj].tolist()
    dBphi_cs_5_36 += dBphi_cs[in_30rj:in_5rj].tolist()
    dBphi_cs_5_36 += dBphi_cs[out_5rj:out_30rj].tolist()
    dBz_cs_5_36 += dBz_cs[in_30rj:in_5rj].tolist()
    dBz_cs_5_36 += dBz_cs[out_5rj:out_30rj].tolist()

Brho_cs_res_5_36 = np.array(Brho_cs_res_5_36)
Bphi_cs_res_5_36 = np.array(Bphi_cs_res_5_36)
Bz_cs_res_5_36 = np.array(Bz_cs_res_5_36)
rho_cs_5_36 = np.array(rho_cs_5_36)
phi_cs_5_36 = np.array(phi_cs_5_36)
z_cs_5_36 = np.array(z_cs_5_36)
JunoMLT_5_36 = np.array(JunoMLT_5_36)
JunoPJ_5_36 = np.array(JunoPJ_5_36)
dBrho_cs_5_36 = np.array(dBrho_cs_5_36)
dBphi_cs_5_36 = np.array(dBphi_cs_5_36)
dBz_cs_5_36 = np.array(dBz_cs_5_36)

# ===========================================================
# Dummy
# ===========================================================
sc = F.ax.scatter(-999.9,
                  -999.9,
                  c=0.0,
                  vmin=-20, vmax=20, cmap=cmap,
                  s=0.6,
                  rasterized=True,)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(F.ax)

# ===========================================================
# Jupiter
# ===========================================================
F.ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                          zorder=6, fill=True, clip_on=True))

# ===========================================================
# Galilean moons
# ===========================================================
draw_Galilean_moons(F.ax, facealpha=0.5)

# ===========================================================
# Hatch mask outside 36 RJ
# ===========================================================
hatch_outer_36rj(F.ax)

dusk_latitude_label_right(F.ax)

# ===========================================================
# Colorbar
# ===========================================================
cax = F.fig.colorbar(sc, ax=F.ax)
cax.ax.set_yticks(np.arange(-20, 20+1, 5))
cax.ax.set_yticklabels(np.arange(-20, 20+1, 5, dtype=int),
                       fontsize=F.fontsize*0.8)
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
cax.ax.set_ylabel(
    r'$\Delta B_{\varphi_{\rm cs}}^{\rm FAC}$ [nT]',
    fontsize=F.fontsize*0.8)

F.ax.set_title('Azimuthal field residuals', weight='bold')

plt.savefig('img/FAC_v2/residual_azimuthal_all_MLT_test.pdf',
            bbox_inches='tight')

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

# ===========================================================
# Plot settings
# ===========================================================
F = ShareXaxis()
F.fontsize = 17
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(6.2, 5.5), dpi='XL')
F.initialize()

F.ax.set_aspect('equal')

F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
            min=0.0, max=40.0,
            ticks=np.arange(0, 45+1, 5),
            ticklabels=np.arange(0, 45+1, 5),
            minor_num=5)
F.set_yaxis(ax_idx=0,
            label=r'$z_{\rm cs}$ [$R_{\rm J}$]',
            min=-40.0, max=15.0,
            ticks=np.arange(-40, 15+1, 5),
            ticklabels=np.arange(-40, 15+1, 5, dtype=int),
            minor_num=5,)
F.ax.set_facecolor('k')
cmap = plt.get_cmap('RdBu_r')

# ===========================================================
# Color coded plot
# ===========================================================
F.ax.scatter(rho_cs_5_36,
             z_cs_5_36,
             c=Brho_cs_res_5_36,
             vmin=-20, vmax=20, cmap=cmap,
             s=0.1,
             rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = F.ax.scatter(-999.9,
                  -999.9,
                  c=0.0,
                  vmin=-20, vmax=20, cmap=cmap,
                  s=0.6,
                  rasterized=True,)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(F.ax)

# ===========================================================
# Jupiter
# ===========================================================
F.ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                          zorder=6, fill=True, clip_on=True))

# ===========================================================
# Galilean moons
# ===========================================================
draw_Galilean_moons(F.ax, facealpha=0.5)

# ===========================================================
# Hatch mask outside 36 RJ
# ===========================================================
hatch_outer_36rj(F.ax)

dusk_latitude_label_right(F.ax)

# ===========================================================
# Colorbar
# ===========================================================
cax = F.fig.colorbar(sc, ax=F.ax)
cax.ax.set_yticks(np.arange(-20, 20+1, 5))
cax.ax.set_yticklabels(np.arange(-20, 20+1, 5, dtype=int),
                       fontsize=F.fontsize*0.8)
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
cax.ax.set_ylabel(
    r'$\Delta B_{\rho_{\rm cs}}^{\rm FAC}$ [nT]',
    fontsize=F.fontsize*0.8)

F.ax.set_title('Radial field residuals', weight='bold')

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

# ===========================================================
# Plot settings
# ===========================================================
F = ShareXaxis()
F.fontsize = 17
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(6.2, 5.5), dpi='XL')
F.initialize()

F.ax.set_aspect('equal')

F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
            min=0.0, max=40.0,
            ticks=np.arange(0, 45+1, 5),
            ticklabels=np.arange(0, 45+1, 5),
            minor_num=5)
F.set_yaxis(ax_idx=0,
            label=r'$z_{\rm cs}$ [$R_{\rm J}$]',
            min=-40.0, max=15.0,
            ticks=np.arange(-40, 15+1, 5),
            ticklabels=np.arange(-40, 15+1, 5, dtype=int),
            minor_num=5,)
F.ax.set_facecolor('k')
cmap = plt.get_cmap('RdBu_r')

# ===========================================================
# Color coded plot
# ===========================================================
F.ax.scatter(rho_cs_5_36,
             z_cs_5_36,
             c=Bz_cs_res_5_36,
             vmin=-20, vmax=20, cmap=cmap,
             s=0.1,
             rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = F.ax.scatter(-999.9,
                  -999.9,
                  c=0.0,
                  vmin=-20, vmax=20, cmap=cmap,
                  s=0.6,
                  rasterized=True,)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(F.ax)

# ===========================================================
# Jupiter
# ===========================================================
F.ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                          zorder=6, fill=True, clip_on=True))

# ===========================================================
# Galilean moons
# ===========================================================
draw_Galilean_moons(F.ax, facealpha=0.5)

# ===========================================================
# Hatch mask outside 36 RJ
# ===========================================================
hatch_outer_36rj(F.ax)

dusk_latitude_label_right(F.ax)

# ===========================================================
# Colorbar
# ===========================================================
cax = F.fig.colorbar(sc, ax=F.ax)
cax.ax.set_yticks(np.arange(-20, 20+1, 5))
cax.ax.set_yticklabels(np.arange(-20, 20+1, 5, dtype=int),
                       fontsize=F.fontsize*0.8)
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
cax.ax.set_ylabel(
    r'$\Delta B_{z_{\rm cs}}^{\rm FAC}$ [nT]',
    fontsize=F.fontsize*0.8)

F.ax.set_title('Axial field residuals', weight='bold')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=Bphi_cs_res_5_36[dawn_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=Bphi_cs_res_5_36[dusk_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\varphi_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Azimuthal field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/residual_azimuthal_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
vmax = 20.0
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=dBphi_cs_5_36[dawn_idx],
           vmin=-vmax, vmax=vmax, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-vmax, vmax=vmax, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=dBphi_cs_5_36[dusk_idx],
           vmin=-vmax, vmax=vmax, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-vmax, vmax=vmax, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{\varphi_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Azimuthal external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/total_external_azimuthal_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=Brho_cs_res_5_36[dawn_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=Brho_cs_res_5_36[dusk_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\rho_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Radial field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/residual_radial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=dBrho_cs_5_36[dawn_idx],
           vmin=-75, vmax=75, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-75, vmax=75, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=dBrho_cs_5_36[dusk_idx],
           vmin=-75, vmax=75, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-75, vmax=75, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-75, 75+1, 25))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{\rho_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Radial external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/total_external_radial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=Bz_cs_res_5_36[dawn_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=Bz_cs_res_5_36[dusk_idx],
           vmin=-20, vmax=20, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{z_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Axial field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/residual_axial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
vmax = 150.0
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
ax.scatter(rho_cs_5_36[dawn_idx],
           z_cs_5_36[dawn_idx],
           c=dBz_cs_5_36[dawn_idx],
           vmin=-vmax, vmax=vmax, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-vmax, vmax=vmax, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')


# ===========================================================
# Color coded plot
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
ax.scatter(rho_cs_5_36[dusk_idx],
           z_cs_5_36[dusk_idx],
           c=dBz_cs_5_36[dusk_idx],
           vmin=-vmax, vmax=vmax, cmap=cmap,
           s=0.2,
           rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-vmax, vmax=vmax, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-150, 150+1, 50))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{z_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Axial external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/total_external_axial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

# PJ01-PJ21 Active v.s. Quiet

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'High\nMidnight-dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
I_rho_median_dawn = np.median(f0[:22, 3])
PJ_select = np.array(PJ_list)[np.where(f0[:22, 3] > I_rho_median_dawn)]
print(PJ_select)
for i in range(PJ_select.size):
    dawn_idx = np.where(JunoPJ_5_36 == PJ_select[i])
    ax.scatter(rho_cs_5_36[dawn_idx],
               z_cs_5_36[dawn_idx],
               c=Bphi_cs_res_5_36[dawn_idx],
               vmin=-20, vmax=20, cmap=cmap,
               s=0.2,
               rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Low\nMidnight-dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

print(JunoPJ_5_36.shape)
print(rho_cs_5_36.shape)

# ===========================================================
# Color coded plot
# ===========================================================
I_rho_median_dawn = np.median(f0[:22, 3])
PJ_select = np.array(PJ_list)[np.where(f0[:22, 3] <= I_rho_median_dawn)]
print(PJ_select)
for i in range(PJ_select.size):
    dawn_idx = np.where(JunoPJ_5_36 == PJ_select[i])
    ax.scatter(rho_cs_5_36[dawn_idx],
               z_cs_5_36[dawn_idx],
               c=Bphi_cs_res_5_36[dawn_idx],
               vmin=-20, vmax=20, cmap=cmap,
               s=0.2,
               rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\varphi_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Azimuthal field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

# PJ22-PJ80 Active v.s. Quiet

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax, facealpha=0.5)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'High\nDuskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)


# ===========================================================
# Color coded plot
# ===========================================================
I_rho_median_dusk = np.median(f0[21:, 3])
PJ_select = (np.array(PJ_list)+21)[np.where(f0[21:, 3] > I_rho_median_dusk)]
print(PJ_select)
for i in range(PJ_select.size):
    dusk_idx = np.where(JunoPJ_5_36 == PJ_select[i])
    ax.scatter(rho_cs_5_36[dusk_idx],
               z_cs_5_36[dusk_idx],
               c=Bphi_cs_res_5_36[dusk_idx],
               vmin=-20, vmax=20, cmap=cmap,
               s=0.2,
               rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Low\nDuskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

print(JunoPJ_5_36.shape)
print(rho_cs_5_36.shape)

# ===========================================================
# Color coded plot
# ===========================================================
I_rho_median_dusk = np.median(f0[21:, 3])
PJ_select = (np.array(PJ_list)+21)[np.where(f0[21:, 3] <= I_rho_median_dusk)]
print(PJ_select)
for i in range(PJ_select.size):
    dusk_idx = np.where(JunoPJ_5_36 == PJ_select[i])
    print(np.max(dusk_idx))
    ax.scatter(rho_cs_5_36[dusk_idx],
               z_cs_5_36[dusk_idx],
               c=Bphi_cs_res_5_36[dusk_idx],
               vmin=-20, vmax=20, cmap=cmap,
               s=0.2,
               rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = ax.scatter(-999.9,
                -999.9,
                c=0.0,
                vmin=-20, vmax=20, cmap=cmap,
                s=0.6,
                rasterized=True,)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)


# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(sc, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\varphi_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Azimuthal field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

# All MLT (0-6 and 12-24)

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

# ===========================================================
# Plot settings
# ===========================================================
F = ShareXaxis()
F.fontsize = 17
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(6.2, 5.5), dpi='XL')
F.initialize()

F.ax.set_aspect('equal')

F.set_xaxis(label=r'$\rho_{\rm cs}$ [$R_{\rm J}$]',
            min=0.0, max=40.0,
            ticks=np.arange(0, 45+1, 5),
            ticklabels=np.arange(0, 45+1, 5),
            minor_num=5)
F.set_yaxis(ax_idx=0,
            label=r'$z_{\rm cs}$ [$R_{\rm J}$]',
            min=-40.0, max=15.0,
            ticks=np.arange(-40, 15+1, 5),
            ticklabels=np.arange(-40, 15+1, 5, dtype=int),
            minor_num=5,)
F.ax.set_facecolor('k')
cmap = plt.get_cmap('RdBu_r')

# ===========================================================
# Color coded plot
# ===========================================================
F.ax.scatter(rho_cs_5_36,
             z_cs_5_36,
             c=JunoMLT_5_36,
             vmin=0, vmax=24, cmap=cmap,
             s=0.01,
             rasterized=True,)

# ===========================================================
# Dummy
# ===========================================================
sc = F.ax.scatter(-999.9,
                  -999.9,
                  c=0.0,
                  vmin=0, vmax=24, cmap=cmap,
                  s=0.6,
                  rasterized=True,)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(F.ax)

# ===========================================================
# Jupiter
# ===========================================================
F.ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                          zorder=6, fill=True, clip_on=True))

# ===========================================================
# Galilean moons
# ===========================================================
draw_Galilean_moons(F.ax)

# ===========================================================
# Hatch mask outside 36 RJ
# ===========================================================
hatch_outer_36rj(F.ax)

# ===========================================================
# Colorbar
# ===========================================================
cax = F.fig.colorbar(sc, ax=F.ax)
cax.ax.set_yticks(np.arange(0, 24+1, 3))
cax.ax.set_yticklabels(np.arange(0, 24+1, 3, dtype=int),
                       fontsize=F.fontsize*0.8)
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(3))
cax.ax.set_ylabel(
    r'Juno MLT [hour]',
    fontsize=F.fontsize*0.8,
    verticalalignment='bottom',
    rotation=270)

F.ax.set_title('Juno MLT', weight='bold')

# FAC Dawnside

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dawn_idx = np.where(JunoMLT_5_36 <= 12.0)
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dawn_idx],
    z_cs_5_36[dawn_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dawn_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dawn_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dawn_idx])

# B_rhoを補間 (total external field component)
dBrho_cs_g = interp(dBrho_cs_5_36[dawn_idx])
dBphi_cs_g = interp(dBphi_cs_5_36[dawn_idx])
dBz_cs_g = interp(dBz_cs_5_36[dawn_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dawn_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dawn_idx], z_cs_5_36[dawn_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan
dBrho_cs_g[outside_mask] = np.nan   # Total external field component
dBphi_cs_g[outside_mask] = np.nan   # Total external field component
dBz_cs_g[outside_mask] = np.nan     # Total external field component


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 3.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_0060 = J_para
RHO_CS_0060 = RHO_CS
Z_CS_0060 = Z_CS
J_rho_grid_0060 = J_rho_grid
J_phi_grid_0060 = J_phi_grid
J_z_grid_0060 = J_z_grid
dBrho_cs_res_g_0060 = dBrho_cs_res_g
dBphi_cs_res_g_0060 = dBphi_cs_res_g
dBz_cs_res_g_0060 = dBz_cs_res_g
dBrho_cs_g_0060 = dBrho_cs_g    # Total external field component
dBphi_cs_g_0060 = dBphi_cs_g    # Total external field component
dBz_cs_g_0060 = dBz_cs_g    # Total external field component
B0_grid_0060 = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')
    ax.tick_params(
        axis='both',       # Apply to both x and y axes
        which='both',      # Apply to both major and minor ticks
        top=True,          # Show top tick marks
        right=True,        # Show right tick marks
    )

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)

    # ===========================================================
    # Magnetic field line
    # ===========================================================
    draw_field_line_0600(ax)

    # ===========================================================
    # Latitude labels on arc of r=40 RJ
    # ===========================================================
    dusk_latitude_label_right(ax)


# ===========================================================
# Left panel
# ===========================================================
ax = axes[0]
ax.set_title(r'$J_\parallel$ (Dawn)',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    J_para_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()
cax = fig.colorbar(cf, ax=ax)
cax.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
cax.ax.set_ylabel(r'$J_\parallel$ [nA m$^{-2}$]',
                  verticalalignment='bottom',
                  rotation=270)


# ===========================================================
# Right panel
# ===========================================================
ax = axes[1]
ax.set_title(r'Inward $J_\parallel$ (Dawn)',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_0060)*J_para_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()
cax = fig.colorbar(cf, ax=ax)
cax.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
cax.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                  verticalalignment='bottom',
                  rotation=270)

fig.tight_layout()

# FAC Duskside

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# B_rhoを補間 (total external field component)
dBrho_cs_g = interp(dBrho_cs_5_36[dusk_idx])
dBphi_cs_g = interp(dBphi_cs_5_36[dusk_idx])
dBz_cs_g = interp(dBz_cs_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan
dBrho_cs_g[outside_mask] = np.nan   # Total external field component
dBphi_cs_g[outside_mask] = np.nan   # Total external field component
dBz_cs_g[outside_mask] = np.nan     # Total external field component


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 18.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1224 = J_para
RHO_CS_1224 = RHO_CS
Z_CS_1224 = Z_CS
J_rho_grid_1224 = J_rho_grid
J_phi_grid_1224 = J_phi_grid
J_z_grid_1224 = J_z_grid
dBrho_cs_res_g_1224 = dBrho_cs_res_g
dBphi_cs_res_g_1224 = dBphi_cs_res_g
dBz_cs_res_g_1224 = dBz_cs_res_g
dBrho_cs_g_1224 = dBrho_cs_g    # Total external field component
dBphi_cs_g_1224 = dBphi_cs_g    # Total external field component
dBz_cs_g_1224 = dBz_cs_g        # Total external field component
B0_grid_1224 = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')
    ax.tick_params(
        axis='both',       # Apply to both x and y axes
        which='both',      # Apply to both major and minor ticks
        top=True,          # Show top tick marks
        right=True,        # Show right tick marks
    )

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)

    # ===========================================================
    # Magnetic field line
    # ===========================================================
    draw_field_line_1800(ax)

    # ===========================================================
    # Latitude labels on arc of r=40 RJ
    # ===========================================================
    dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Left panel
# ===========================================================
ax = axes[0]
ax.set_title(r'$J_\parallel$ (Dusk)',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    J_para_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()
cax = fig.colorbar(cf, ax=ax)
cax.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
cax.ax.set_ylabel(r'$J_\parallel$ [nA m$^{-2}$]',
                  verticalalignment='bottom',
                  rotation=270)


# ===========================================================
# Right panel
# ===========================================================
ax = axes[1]
ax.set_title(r'Inward $J_\parallel$ (Dusk)',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224)*J_para_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()
cax = fig.colorbar(cf, ax=ax)
cax.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
cax.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
cax.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                  verticalalignment='bottom',
                  rotation=270)

fig.tight_layout()

In [ ]:
# Dusk側のデータ点を南半球高緯度領域に絞って確認
dusk_sparse_idx = np.where(
    (JunoMLT_5_36 > 12.0) &
    (z_cs_5_36 < -15.0) &
    (rho_cs_5_36 < 10.0)
)[0]

print(f"該当データ点数: {len(dusk_sparse_idx)}")
print(
    f"rho_cs範囲: {rho_cs_5_36[dusk_sparse_idx].min():.1f} – {rho_cs_5_36[dusk_sparse_idx].max():.1f} RJ")
print(
    f"z_cs範囲: {z_cs_5_36[dusk_sparse_idx].min():.1f} – {z_cs_5_36[dusk_sparse_idx].max():.1f} RJ")

# FAC Dusk 12-18 & 18-24

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dusk_idx = np.where((JunoMLT_5_36 > 12.0) & (JunoMLT_5_36 <= 18.0))
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1218 = J_para
RHO_CS_1218 = RHO_CS
Z_CS_1218 = Z_CS
J_rho_grid_1218 = J_rho_grid
J_phi_grid_1218 = J_phi_grid
J_z_grid_1218 = J_z_grid
dBrho_cs_res_g_1218 = dBrho_cs_res_g
dBphi_cs_res_g_1218 = dBphi_cs_res_g
dBz_cs_res_g_1218 = dBz_cs_res_g
B0_grid_1218 = B0_grid

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 18.0)
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 21.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1824 = J_para
RHO_CS_1824 = RHO_CS
Z_CS_1824 = Z_CS
J_rho_grid_1824 = J_rho_grid
J_phi_grid_1824 = J_phi_grid
J_z_grid_1824 = J_z_grid
dBrho_cs_res_g_1824 = dBrho_cs_res_g
dBphi_cs_res_g_1824 = dBphi_cs_res_g
dBz_cs_res_g_1824 = dBz_cs_res_g
B0_grid_1824 = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)

    # ===========================================================
    # Magnetic field line
    # ===========================================================
    draw_field_line_1800(ax)

    # ===========================================================
    # Latitude labels on arc of r=40 RJ
    # ===========================================================
    dusk_latitude_label_right(ax)


# ===========================================================
# Left panel (12-18)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Noon-Dusk\n12-18 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1218)*J_para_1218,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1218.min(), RHO_CS_1218.max(),
        Z_CS_1218.min(),   Z_CS_1218.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Dusk-Midnight\n18-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1824)*J_para_1824,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1824.min(), RHO_CS_1824.max(),
        Z_CS_1824.min(),   Z_CS_1824.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/FAC_1218MLT_vs_1824MLT.pdf',
            bbox_inches='tight')

# FAC Dawn/Dusk Side by Side

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_0060)*J_para_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224)*J_para_1224,
    cmap='RdBu_r',
    # cmap=cmap_bwr,
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/FAC_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_0060)*J_para_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Dusk-Midnight\n18-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1824)*J_para_1824,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1824.min(), RHO_CS_1824.max(),
        Z_CS_1824.min(),   Z_CS_1824.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/FAC_0060MLT_vs_1824MLT.pdf',
            bbox_inches='tight')

# FAC Dusk 15-21

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dusk_idx = np.where((JunoMLT_5_36 > 15.0) & (JunoMLT_5_36 <= 21.0))
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1521 = J_para
RHO_CS_1521 = RHO_CS
Z_CS_1521 = Z_CS
J_rho_grid_1521 = J_rho_grid
J_phi_grid_1521 = J_phi_grid
J_z_grid_1521 = J_z_grid
dBrho_cs_res_g_1521 = dBrho_cs_res_g
dBphi_cs_res_g_1521 = dBphi_cs_res_g
dBz_cs_res_g_1521 = dBz_cs_res_g
B0_grid_1521 = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Dusk\n15-21 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1521)*J_para_1521,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1521.min(), RHO_CS_1521.max(),
        Z_CS_1521.min(),   Z_CS_1521.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Dusk-Midnight\n18-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1824)*J_para_1824,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1824.min(), RHO_CS_1824.max(),
        Z_CS_1824.min(),   Z_CS_1824.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/FAC_1521MLT_vs_1824MLT.pdf',
            bbox_inches='tight')

# FAC in PJ01-PJ21 Active v.s. Quiet

In [ ]:
# ===========================================================
# Create a grid
# ===========================================================
d_rj = 1.25
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)

points = np.zeros((3, 2))
dawn_idx = []
I_rho_median_dusk = np.median(f0[:22, 3])
PJ_select = np.array(PJ_list)[np.where(f0[:22, 3] > I_rho_median_dusk)]
for i in range(PJ_select.size):
    dawn_idx_0 = np.where(JunoPJ_5_36 == PJ_select[i])
    dawn_idx += dawn_idx_0[0].tolist()

points = np.column_stack([
    rho_cs_5_36[dawn_idx],
    z_cs_5_36[dawn_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


# B^FACを補間
dBrho_cs_res_g = interp(Brho_cs_res_5_36[dawn_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dawn_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dawn_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dawn_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dawn_idx], z_cs_5_36[dawn_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 1.25)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 2.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_0060_high = J_para
RHO_CS_0060_high = RHO_CS
Z_CS_0060_high = Z_CS
J_rho_grid_0060_high = J_rho_grid
J_phi_grid_0060_high = J_phi_grid
J_z_grid_0060_high = J_z_grid
dBrho_cs_res_g_0060_high = dBrho_cs_res_g
dBphi_cs_res_g_0060_high = dBphi_cs_res_g
dBz_cs_res_g_0060_high = dBz_cs_res_g
B0_grid_0060_high = B0_grid

In [ ]:
# ===========================================================
# Create a grid
# ===========================================================
d_rj = 1.25
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)

points = np.zeros((3, 2))
dawn_idx = []
I_rho_median_dusk = np.median(f0[:22, 3])
PJ_select = np.array(PJ_list)[np.where(f0[:22, 3] <= I_rho_median_dusk)]
for i in range(PJ_select.size):
    dawn_idx_0 = np.where(JunoPJ_5_36 == PJ_select[i])
    dawn_idx += dawn_idx_0[0].tolist()

points = np.column_stack([
    rho_cs_5_36[dawn_idx],
    z_cs_5_36[dawn_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


dBrho_cs_res_g = interp(Brho_cs_res_5_36[dawn_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dawn_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dawn_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dawn_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dawn_idx], z_cs_5_36[dawn_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 1.25)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 2.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_0060_low = J_para
RHO_CS_0060_low = RHO_CS
Z_CS_0060_low = Z_CS
J_rho_grid_0060_low = J_rho_grid
J_phi_grid_0060_low = J_phi_grid
J_z_grid_0060_low = J_z_grid
dBrho_cs_res_g_0060_low = dBrho_cs_res_g
dBphi_cs_res_g_0060_low = dBphi_cs_res_g
dBz_cs_res_g_0060_low = dBz_cs_res_g
B0_grid_0060_low = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'High\nMidnight-dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_0060_high)*J_para_0060_high,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060_high.min(), RHO_CS_0060_high.max(),
        Z_CS_0060_high.min(),   Z_CS_0060_high.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Low\nMidnight-dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_0060_low)*J_para_0060_low,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060_low.min(), RHO_CS_0060_low.max(),
        Z_CS_0060_low.min(),   Z_CS_0060_low.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

# plt.savefig('img/FAC_v2/FAC_1521MLT_vs_1824MLT.pdf',
#             bbox_inches='tight')

# FAC in PJ22-PJ80 Active v.s. Quiet

In [ ]:
# ===========================================================
# Create a grid
# ===========================================================
d_rj = 1.0
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)

points = np.zeros((3, 2))
dusk_idx = []
I_rho_median_dusk = np.median(f0[21:, 3])
PJ_select = (np.array(PJ_list)+21)[np.where(f0[21:, 3] > I_rho_median_dusk)]
for i in range(PJ_select.size):
    dusk_idx_0 = np.where(JunoPJ_5_36 == PJ_select[i])
    dusk_idx += dusk_idx_0[0].tolist()

points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 1.0)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 2.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1224_high = J_para
RHO_CS_1224_high = RHO_CS
Z_CS_1224_high = Z_CS
J_rho_grid_1224_high = J_rho_grid
J_phi_grid_1224_high = J_phi_grid
J_z_grid_1224_high = J_z_grid
dBrho_cs_res_g_1224_high = dBrho_cs_res_g
dBphi_cs_res_g_1224_high = dBphi_cs_res_g
dBz_cs_res_g_1224_high = dBz_cs_res_g
B0_grid_1224_high = B0_grid

In [ ]:
# ===========================================================
# Create a grid
# ===========================================================
d_rj = 1.0
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)

points = np.zeros((3, 2))
dusk_idx = []
I_rho_median_dusk = np.median(f0[21:, 3])
PJ_select = (np.array(PJ_list)+21)[np.where(f0[21:, 3] <= I_rho_median_dusk)]
for i in range(PJ_select.size):
    dusk_idx_0 = np.where(JunoPJ_5_36 == PJ_select[i])
    dusk_idx += dusk_idx_0[0].tolist()

points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 1.0)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 2.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)
B0_grid = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 15.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

            B0_grid[iz, ir] = B0

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1224_low = J_para
RHO_CS_1224_low = RHO_CS
Z_CS_1224_low = Z_CS
J_rho_grid_1224_low = J_rho_grid
J_phi_grid_1224_low = J_phi_grid
J_z_grid_1224_low = J_z_grid
dBrho_cs_res_g_1224_low = dBrho_cs_res_g
dBphi_cs_res_g_1224_low = dBphi_cs_res_g
dBz_cs_res_g_1224_low = dBz_cs_res_g
B0_grid_1224_low = B0_grid

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'High\nDuskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224_high)*J_para_1224_high,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_high.min(), RHO_CS_1224_high.max(),
        Z_CS_1224_high.min(),   Z_CS_1224_high.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Low\nDuskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224_low)*J_para_1224_low,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_low.min(), RHO_CS_1224_low.max(),
        Z_CS_1224_low.min(),   Z_CS_1224_low.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

fig.suptitle('Inward FAC density', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

# plt.savefig('img/FAC_v2/FAC_1521MLT_vs_1824MLT.pdf',
#             bbox_inches='tight')

# Magnetic field residuals (pixelized)

In [ ]:
pj01_trajectory = np.loadtxt(
    'results/Juno_trajectory/PJ01/pos_iau.txt')  # [RJ]
x_pj01 = pj01_trajectory[0, :]
y_pj01 = pj01_trajectory[1, :]
z_pj01 = pj01_trajectory[2, :]

x_pj01_cs, y_pj01_cs, z_pj01_cs = csfield._sys3_2_cs(x_pj01, y_pj01, z_pj01)
rho_pj01_cs = np.sqrt(x_pj01_cs**2+y_pj01_cs**2)

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBrho_cs_res_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)

# ===========================================================
# Selected PJ trajectory
# ===========================================================
ax.plot(rho_pj01_cs, z_pj01_cs,
        color=UC.purple, linewidth=0.75, zorder=5.0)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBrho_cs_res_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\rho_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Radial field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/B_residual_radial.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 75.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBrho_cs_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)

# ===========================================================
# Selected PJ trajectory
# ===========================================================
ax.plot(rho_pj01_cs, z_pj01_cs,
        color=UC.purple, linewidth=0.85, zorder=5.0)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 75.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBrho_cs_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-75, 75+1, 25))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{\rho_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)


# ===========================================================
# Title & panel names
# ===========================================================
fig.suptitle('Radial external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")
F.panelname_fontsize = F.fontsize
F.panelname_xposition = 'left'
F.draw_panel_name(ax=axes[0], panelname=' (a) ', background='none')
F.panelname_xposition = 'right'
F.draw_panel_name(ax=axes[1], panelname=' (b) ', background='none', a=-1)

plt.savefig('img/FAC_v2/total_external_grid_radial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBphi_cs_res_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBphi_cs_res_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{\varphi_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Azimuthal field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/B_residual_azimuthal.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBphi_cs_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)

# ===========================================================
# Selected PJ trajectory
# ===========================================================
ax.plot(rho_pj01_cs, z_pj01_cs,
        color=UC.purple, linewidth=0.85, zorder=5.0)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBphi_cs_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{\varphi_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)


# ===========================================================
# Title & panel names
# ===========================================================
fig.suptitle('Azimuthal external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")
F.panelname_fontsize = F.fontsize
F.panelname_xposition = 'left'
F.draw_panel_name(ax=axes[0], panelname=' (a) ', background='none')
F.panelname_xposition = 'right'
F.draw_panel_name(ax=axes[1], panelname=' (b) ', background='none', a=-1)

plt.savefig('img/FAC_v2/total_external_grid_azimuthal_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBz_cs_res_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 20.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBz_cs_res_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-20, 20+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\Delta B_{z_{\rm cs}}^{\rm FAC}$ [nT]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle('Axial field component residuals', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/B_residual_axial.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vlim = 150.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBz_cs_g_0060,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)

# ===========================================================
# Selected PJ trajectory
# ===========================================================
ax.plot(rho_pj01_cs, z_pj01_cs,
        color=UC.purple, linewidth=0.85, zorder=5.0)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vlim = 150.0
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    dBz_cs_g_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(-150, 150+1, 50))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\delta B_{z_{\rm cs}}^{\rm obs}$ [nT]',
    verticalalignment='bottom',
    rotation=270)


# ===========================================================
# Title & panel names
# ===========================================================
fig.suptitle('Axial external field component', y=1.01,
             fontsize=F.fontsize, fontweight="bold")
F.panelname_fontsize = F.fontsize
F.panelname_xposition = 'left'
F.draw_panel_name(ax=axes[0], panelname=' (a) ', background='none')
F.panelname_xposition = 'right'
F.draw_panel_name(ax=axes[1], panelname=' (b) ', background='none', a=-1)

plt.savefig('img/FAC_v2/total_external_grid_axial_0060MLT_vs_1224MLT.pdf',
            bbox_inches='tight')

# Dusk radial residual test

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

# ===========================================================
# Create a grid
# ===========================================================
dusk_idx = np.where(JunoMLT_5_36 > 12.0)
d_rj = 0.5
grid_rho_cs = np.arange(1.0, 45.0+1, d_rj)
grid_z_cs = np.arange(-40.0, 15.0+1, d_rj)
RHO_CS, Z_CS = np.meshgrid(grid_rho_cs, grid_z_cs)   # shape (Nz, Nrho)
points = np.column_stack([
    rho_cs_5_36[dusk_idx],
    z_cs_5_36[dusk_idx]])  # shape (N, 2)


# ----------------------------------------------------------
# 1. 散布点 → 2D グリッドに補間
# ----------------------------------------------------------
def interp(values):
    grid = griddata(points, values, (RHO_CS, Z_CS), method='linear')
    grid_nn = griddata(points, values, (RHO_CS, Z_CS), method='nearest')
    mask = np.isnan(grid)
    grid[mask] = grid_nn[mask]
    return grid


dBrho_cs_res_g = interp(Brho_cs_res_5_36[dusk_idx])
dBphi_cs_res_g = interp(Bphi_cs_res_5_36[dusk_idx])
dBz_cs_res_g = interp(Bz_cs_res_5_36[dusk_idx])

# カバレッジマスク
_ones = np.ones(len(rho_cs_5_36[dusk_idx]))
mask_coverage = griddata(points, _ones, (RHO_CS, Z_CS), method='linear')
outside_mask = np.isnan(mask_coverage)

# KDTreeマスク
tree = KDTree(np.column_stack([rho_cs_5_36[dusk_idx], z_cs_5_36[dusk_idx]]))
nn_dist, _ = tree.query(np.column_stack([RHO_CS.ravel(), Z_CS.ravel()]))
nn_dist_grid = nn_dist.reshape(RHO_CS.shape)
outside_mask = outside_mask | (nn_dist_grid > 0.5)

# 平滑化の前にマスク領域をnanに戻す
dBrho_cs_res_g[outside_mask] = np.nan
dBphi_cs_res_g[outside_mask] = np.nan
dBz_cs_res_g[outside_mask] = np.nan


# ----------------------------------------------------------
# 2. 平滑化（マスク境界を考慮）
# ----------------------------------------------------------
smooth_sigma = 1.0 / d_rj


def smooth_with_mask(data, sigma, mask):
    data_filled = data.copy()
    data_filled[mask] = 0.0
    smoothed = gaussian_filter(data_filled, sigma=sigma)
    weight = np.ones_like(data)
    weight[mask] = 0.0
    weight_smoothed = gaussian_filter(weight, sigma=sigma)
    return np.where(weight_smoothed > 0.01, smoothed / weight_smoothed, 0.0)


dBrho_g = smooth_with_mask(dBrho_cs_res_g, smooth_sigma, outside_mask)
dBphi_g = smooth_with_mask(dBphi_cs_res_g, smooth_sigma, outside_mask)
dBz_g = smooth_with_mask(dBz_cs_res_g,   smooth_sigma, outside_mask)


# ----------------------------------------------------------
# 3. 数値微分: ∂/∂ρ, ∂/∂z  [nT/RJ]
# ----------------------------------------------------------
drho = np.gradient(grid_rho_cs)   # shape (Nrho,)
dz = np.gradient(grid_z_cs)     # shape (Nz,)

# np.gradient は各軸方向に微分: axis=1 → ρ方向, axis=0 → z方向
# dBphi_g: shape (Nz, Nrho)

# ∂(dBφ)/∂z  [nT/RJ]
ddBphi_dz = np.gradient(dBphi_g,  grid_z_cs,   axis=0)
# ∂(dBφ)/∂ρ  [nT/RJ]
ddBphi_drho = np.gradient(dBphi_g, grid_rho_cs, axis=1)
# ∂(ρ·dBφ)/∂ρ = dBφ + ρ·∂dBφ/∂ρ   [nT/RJ]
rho_ddBphi_drho = dBphi_g + RHO_CS * ddBphi_drho

# ∂(dBρ)/∂z  [nT/RJ]
ddBrho_dz = np.gradient(dBrho_g,  grid_z_cs,   axis=0)
ddBrho_dz *= 0.0    # テスト用!!!
print(ddBrho_dz)
# ∂(dBz)/∂ρ  [nT/RJ]
ddBz_drho = np.gradient(dBz_g,    grid_rho_cs, axis=1)


# ----------------------------------------------------------
# 4. 電流密度 J = rot(dB) / μ0   (軸対称仮定: ∂/∂φ = 0)
# ----------------------------------------------------------
# 単位変換: nT/RJ → T/m = nT/RJ * 1e-9 / (RJ_in_m)
# J [A/m²] = (1/μ0) * (nT/RJ) * 1e-9 / RJ_m
conv = 1E-9 / (RJ * MU0)   # [A/m² per (nT/RJ)]

# J_ρ = -(1/μ0) ∂(dBφ)/∂z
J_rho_grid = -ddBphi_dz * conv               # [A/m²]

# J_φ = (1/μ0) (∂dBρ/∂z - ∂dBz/∂ρ)
J_phi_grid = (ddBrho_dz - ddBz_drho) * conv  # [A/m²]
J_phi_grid = -ddBz_drho * conv  # [A/m²]

# J_z = (1/μ₀ρ) ∂(ρ·dBφ)/∂ρ
# ρ=0 の特異点を避ける (ρ > 0.5 RJ)
rho_safe = np.where(RHO_CS > 0.5, RHO_CS, 0.5)
J_z_grid = rho_ddBphi_drho / rho_safe * conv  # [A/m²]

# nA/m² に変換 (見やすい単位)
J_rho_grid *= 1E+9
J_phi_grid *= 1E+9
J_z_grid *= 1E+9


# ----------------------------------------------------------
# 5. 背景磁場方向 B̂ を取得して J∥ を計算
# ----------------------------------------------------------
jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)
Nz_g, Nrho_g = RHO_CS.shape
b_rho = np.zeros(RHO_CS.shape)
b_phi = np.zeros(RHO_CS.shape)
b_z = np.zeros(RHO_CS.shape)

# R: SIII直交 → CS直交
theta_d = math.radians(csfield.theta_d_deg)
phi_d = math.radians(csfield.phi_d_deg)
R = np.array([
    [math.cos(theta_d)*math.cos(phi_d),  math.cos(theta_d)
     * math.sin(phi_d), -math.sin(theta_d)],
    [-math.sin(phi_d),
     math.cos(phi_d),                    0.0],
    [math.sin(theta_d)*math.cos(phi_d),  math.sin(theta_d)
     * math.sin(phi_d),  math.cos(theta_d)]
])

# RT: CS直交 → SIII直交 (Rの転置 = Rの逆行列、直交行列なので)
RT = R.T

phi_0 = math.radians(360.0-112.0)

for iz in range(Nz_g):
    for ir in range(Nrho_g):
        rho_0 = RHO_CS[iz, ir]
        z_0 = Z_CS[iz, ir]

        x_cs = rho_0 * math.cos(phi_0)
        y_cs = rho_0 * math.sin(phi_0)
        z_cs_val = z_0

        pos_siii = RT @ np.array([x_cs, y_cs, z_cs_val])
        x0, y0, z0 = pos_siii[0], pos_siii[1], pos_siii[2]

        try:
            Bx0, By0, Bz0 = jm.Internal.Field(x0, y0, z0)            # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(x0, y0, z0, 18.0)  # [T]

            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]
            B0 = math.sqrt(Bx**2+By**2+Bz**2)      # [T]

            # SIII直交 → CS直交 (回転行列 R を適用)
            B_siii = np.array([Bx/B0, By/B0, Bz/B0])
            B_cs_cart = R @ B_siii   # CS直交成分

            # CS直交 → CS円柱 (phi_0 は CS座標でのφ角)
            B_rho_cs = B_cs_cart[0] * \
                math.cos(phi_0) + B_cs_cart[1]*math.sin(phi_0)
            B_phi_cs = -B_cs_cart[0] * \
                math.sin(phi_0) + B_cs_cart[1]*math.cos(phi_0)
            B_z_cs = B_cs_cart[2]

            b_rho[iz, ir] = B_rho_cs
            b_phi[iz, ir] = B_phi_cs
            b_z[iz, ir] = B_z_cs

        except:
            b_rho[iz, ir], b_phi[iz, ir], b_z[iz, ir] = np.nan, np.nan, np.nan

J_para = J_rho_grid*b_rho + J_phi_grid*b_phi + J_z_grid*b_z

# データカバレッジ外をマスク
J_para[outside_mask] = np.nan
J_rho_grid[outside_mask] = np.nan
J_phi_grid[outside_mask] = np.nan
J_z_grid[outside_mask] = np.nan

J_para_1224_0 = J_para
RHO_CS_1224_0 = RHO_CS
Z_CS_1224_0 = Z_CS
J_rho_grid_1224_0 = J_rho_grid
J_phi_grid_1224_0 = J_phi_grid
J_z_grid_1224_0 = J_z_grid
dBrho_cs_res_g_1224_0 = dBrho_cs_res_g
dBphi_cs_res_g_1224_0 = dBphi_cs_res_g
dBz_cs_res_g_1224_0 = dBz_cs_res_g

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 3, figsize=(21, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(3):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')
    ax.tick_params(
        axis='both',       # Apply to both x and y axes
        which='both',      # Apply to both major and minor ticks
        top=True,          # Show top tick marks
        right=True,        # Show right tick marks
    )

    ax.text(0.95, 0.02,
            'Duskside\n12-24 MLT',
            color=UC.lightbeige,
            fontsize=F.fontsize,
            verticalalignment='bottom',
            horizontalalignment='right',
            transform=ax.transAxes,
            zorder=10)

    # ===========================================================
    # Magnetic field line
    # ===========================================================
    draw_field_line_1800(ax)

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)

    # ===========================================================
    # Latitude labels on arc of r=40 RJ
    # ===========================================================
    dusk_latitude_label_right(ax)

# ===========================================================
# Left panel
# ===========================================================
ax = axes[0]
ax.set_title(r'(a) With radial residuals',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224)*J_para_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Middle panel
# ===========================================================
ax = axes[1]
ax.set_title(r'(b) Without radial residuals',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224_0)*J_para_1224_0,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_0.min(), RHO_CS_1224_0.max(),
        Z_CS_1224_0.min(),   Z_CS_1224_0.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Right panel
# ===========================================================
ax = axes[2]
ax.set_title(r'(c) Difference between a and b',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.01
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf2 = ax.imshow(
    -np.sign(Z_CS_1224)*J_para_1224 -
    (-np.sign(Z_CS_1224_0)*J_para_1224_0),
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_0.min(), RHO_CS_1224_0.max(),
        Z_CS_1224_0.min(),   Z_CS_1224_0.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

ax = axes[0]
axpos = ax.get_position()
cax = fig.add_axes([axpos.x0-axpos.x1*0.25, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    left=True,         # Show left tick marks
    right=False,
)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'(a, b) Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',)
pp.ax.yaxis.set_label_position('left')
pp.ax.yaxis.set_ticks_position('left')


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

ax = axes[2]
axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.02, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf2, cax=cax)
pp.ax.set_yticks(np.linspace(-0.01, 0.01, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'(c) Difference [nA m$^{-2}$]',
                 verticalalignment='bottom',
                 rotation=270)

plt.savefig('img/FAC_v2/FAC_radial_residual_difference_test.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 3, figsize=(21, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(3):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')
    ax.tick_params(
        axis='both',       # Apply to both x and y axes
        which='both',      # Apply to both major and minor ticks
        top=True,          # Show top tick marks
        right=True,        # Show right tick marks
    )

    ax.text(0.95, 0.02,
            'Duskside\n12-24 MLT',
            color=UC.lightbeige,
            fontsize=F.fontsize,
            verticalalignment='bottom',
            horizontalalignment='right',
            transform=ax.transAxes,
            zorder=10)

    # ===========================================================
    # Magnetic field line
    # ===========================================================
    draw_field_line_1800(ax)

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)

    # ===========================================================
    # Latitude labels on arc of r=40 RJ
    # ===========================================================
    dusk_latitude_label_right(ax)

# ===========================================================
# Left panel
# ===========================================================
ax = axes[0]
ax.set_title(r'(a) With radial residuals',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224)*J_para_1224,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Middle panel
# ===========================================================
ax = axes[1]
ax.set_title(r'(b) Without radial residuals',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 0.03
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf = ax.imshow(
    -np.sign(Z_CS_1224_0)*J_para_1224_0,
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_0.min(), RHO_CS_1224_0.max(),
        Z_CS_1224_0.min(),   Z_CS_1224_0.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Right panel
# ===========================================================
ax = axes[2]
ax.set_title(r'(c) Relative difference between a and b',
             fontweight='bold',
             fontsize=F.fontsize,
             y=1.03)

vlim = 100
clim = (-vlim, vlim)

norm = mplcolors.TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
cf2 = ax.imshow(
    100*(-np.sign(Z_CS_1224)*J_para_1224 -
         (-np.sign(Z_CS_1224_0)*J_para_1224_0))/(-np.sign(Z_CS_1224)*J_para_1224),
    cmap='RdBu_r',
    norm=norm,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224_0.min(), RHO_CS_1224_0.max(),
        Z_CS_1224_0.min(),   Z_CS_1224_0.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

ax = axes[0]
axpos = ax.get_position()
cax = fig.add_axes([axpos.x0-axpos.x1*0.25, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    left=True,         # Show left tick marks
    right=False,
)
pp.ax.set_yticks(np.linspace(-0.03, 0.03, 7))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'(a, b) Inward $J_\parallel$ [nA m$^{-2}$]',
                 verticalalignment='bottom',)
pp.ax.yaxis.set_label_position('left')
pp.ax.yaxis.set_ticks_position('left')


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

ax = axes[2]
axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.02, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf2, cax=cax)
pp.ax.set_yticks(np.linspace(-100, 100, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(10))
pp.ax.set_ylabel(r'(c) Relative difference [%]',
                 verticalalignment='bottom',
                 rotation=270)

plt.savefig('img/FAC_v2/FAC_radial_residual_difference_test2.pdf',
            bbox_inches='tight')

In [ ]:
np.nanmax(-np.sign(Z_CS_1224)*J_para_1224 -
          (-np.sign(Z_CS_1224_0)*J_para_1224_0))

# Field line trace from the grid

In [ ]:
grid_tracing = False

In [ ]:
jm.Internal.Config(Model='jrm33', Degree=1, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')
csfield = CSField()
csfield.config(
    i_phi0=4.9213685925554586E-5,
    I_rho0=11.103883377723957,
    a0=3.46781481E-6,
    MLT0=0.52113394,
    a1=18.34895438,
    MLT1=20.0143857,
    b1=4.2518,
    MLTb=-0.2086,
)

In [ ]:
if grid_tracing:
    jm.Internal.Config(Model='jrm33', Degree=1, CartesianIn=True,
                       CartesianOut=True)
    jm.Con2020.Config(equation_type='integral')
    csfield = CSField()
    csfield.config(
        i_phi0=4.9213685925554586E-5,
        I_rho0=11.103883377723957,
        a0=3.46781481E-6,
        MLT0=0.52113394,
        a1=18.34895438,
        MLT1=20.0143857,
        b1=4.2518,
        MLTb=-0.2086,
    )
    # ==============================================================
    # FIELD LINE TRACE IN THE CURRENT SHEET PLANE (DUSK)
    # FROM THR MAGNETOSPHERE
    # ==============================================================
    rho_cs = np.array([10.0, 15.0])*RJ  # [m]
    phi_cs = 0.0      # [rad]
    z_cs = 1.0*RJ     # [m]

    x_cs0 = rho_cs*np.cos(phi_cs)
    y_cs0 = rho_cs*np.sin(phi_cs)
    z_cs0 = z_cs*np.ones(x_cs0.shape)

    x0, y0, z0, = csfield._cs_2_sys3(
        x_cs0, y_cs0, z_cs0,
        np.sqrt(x_cs0**2+y_cs0**2),
        phi_cs
    )   # [m]

    Niter = 45000

    ds0 = 600000.0   # [m]
    p = 1.2

    i_foot_N = np.zeros((x0.size, 3, int(Niter/10)))

    R = csfield.R_sys3_2_cs()
    RT = csfield.R_cs_2_sys3()

    # 北半球電離圏に向かう向き
    ns = -1     # -1: north; 1: south
    for j in range(x0.size):
        k = 0
        x, y, z = x0[j], y0[j], z0[j]
        for i in range(Niter):
            Bx0, By0, Bz0 = jm.Internal.Field(
                x/RJ, y/RJ, z/RJ)       # [nT]
            Bx1, By1, Bz1 = csfield.magnetic_field_v2(
                x/RJ, y/RJ, z/RJ,
                MLT=18.0)  # [T]
            Bx = (Bx0*1E-9)+Bx1     # [T]
            By = (By0*1E-9)+By1     # [T]
            Bz = (Bz0*1E-9)+Bz1     # [T]

            # Sys III -> CS
            B_vec_cs = R @ np.array([Bx, By, Bz])
            x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
            rho_cs = np.sqrt(x_cs**2 + y_cs**2)
            phi_cs = np.arctan2(y_cs, x_cs)
            B_rho_cs = B_vec_cs[0]*np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
            B_phi_cs = 0.0     # rho_cs - z_cs plane
            B_z_cs = B_vec_cs[2]
            B_vec_cs = np.array([
                B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
                B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
                B_z_cs
            ])

            # CS -> Sys III
            B_vec_s3 = RT @ B_vec_cs
            B0 = math.sqrt(B_vec_s3[0, 0]**2+B_vec_s3[1, 0]
                           ** 2+B_vec_s3[2, 0]**2)      # [T]

            ds = ds0/(np.log10(B0*1E+9))**p      # [m]

            # 座標更新 (x, y, z)
            x += (ds*B_vec_s3[0, 0]/B0)*ns
            y += (ds*B_vec_s3[1, 0]/B0)*ns
            z += (ds*B_vec_s3[2, 0]/B0)*ns

            # 座標更新 (r, theta, phi)
            rs = math.sqrt(x**2 + y**2 + z**2)
            theta = math.acos(z/rs)
            phi = math.atan2(y, x)

            if i % 10 == 0:
                i_foot_N[j, 0, k] = x_cs
                i_foot_N[j, 1, k] = y_cs
                i_foot_N[j, 2, k] = z_cs
                k += 1

            # 終了条件 (北の外側境界)
            if z > 55.0*RJ:
                print('North edge [RJ]:')
                i_foot_N[j, 0, k:] = np.nan
                i_foot_N[j, 1, k:] = np.nan
                i_foot_N[j, 2, k:] = np.nan
                break
            # 終了条件 (水平距離の外側境界)
            if math.sqrt(x**2+y**2) > 60.0*RJ:
                print('Horizontal edge')
                i_foot_N[j, 0, k:] = np.nan
                i_foot_N[j, 1, k:] = np.nan
                i_foot_N[j, 2, k:] = np.nan
                break
            # 終了条件 (水平距離の外側境界)
            if math.sqrt(x**2+y**2+z**2) < (66854.0+500.0)*1E+3:
                print('Jupiter surface')
                i_foot_N[j, 0, k:] = np.nan
                i_foot_N[j, 1, k:] = np.nan
                i_foot_N[j, 2, k:] = np.nan
                break

        print(rs/RJ, y_cs/RJ)

In [ ]:
if grid_tracing:
    jm.Internal.Config(Model='jrm33', Degree=1, CartesianIn=True,
                       CartesianOut=True)
    jm.Con2020.Config(equation_type='integral')
    csfield = CSField()
    csfield.config(
        i_phi0=4.9213685925554586E-5,
        I_rho0=11.103883377723957,
        a0=3.46781481E-6,
        MLT0=0.52113394,
        a1=18.34895438,
        MLT1=20.0143857,
        b1=4.2518,
        MLTb=-0.2086,
    )
    # ==============================================================
    # FIELD LINE TRACE IN THE CURRENT SHEET PLANE (1800 MLT DUSK)
    # FROM THR MAGNETOSPHERE
    # ==============================================================
    rho_cs0 = RHO_CS_1224*RJ  # [m]
    phi_cs0 = 0.0             # [rad]
    z_cs0 = Z_CS_1224*RJ      # [m]

    x_cs0 = rho_cs0*np.cos(phi_cs0)     # [m]
    y_cs0 = rho_cs0*np.sin(phi_cs0)     # [m]

    R = csfield.R_sys3_2_cs()
    RT = csfield.R_cs_2_sys3()

    Niter = 50000

    ds0 = 500000.0   # [m]
    p = 1.2

    i_foot_N = np.zeros(x_cs0.shape)
    i_distance_N = np.zeros(x_cs0.shape)

    # 北半球電離圏に向かう向き
    # -1: north; 1: south
    ns = -1
    for k in range(x_cs0.shape[1]):
        for j in range(x_cs0.shape[0]):
            s = 0
            pos_s3_0 = RT @ np.array([
                x_cs0[j, k], y_cs0[j, k], z_cs0[j, k]
            ])
            x = pos_s3_0[0].copy()
            y = pos_s3_0[1].copy()
            z = pos_s3_0[2].copy()
            if z_cs0[j, k] >= 0.0*RJ:
                ns = -1
            elif z_cs0[j, k] < 0.0*RJ:
                ns = -1
                pos_s3_0 = RT @ np.array([
                    x_cs0[j, k], y_cs0[j, k], -z_cs0[j, k]
                ])
                x = pos_s3_0[0].copy()
                y = pos_s3_0[1].copy()
                z = pos_s3_0[2].copy()
            if np.isnan(J_para_1224[j, k]):
                i_foot_N[j, k] = np.nan
                i_distance_N[j, k] = np.nan
                # print('NaN')
                continue
            else:
                for i in range(Niter):
                    Bx0, By0, Bz0 = jm.Internal.Field(
                        x/RJ, y/RJ, z/RJ)       # [nT]
                    Bx1, By1, Bz1 = csfield.magnetic_field_v2(
                        x/RJ, y/RJ, z/RJ,
                        MLT=18.0)  # [T]
                    Bx = (Bx0*1E-9)+Bx1     # [T]
                    By = (By0*1E-9)+By1     # [T]
                    Bz = (Bz0*1E-9)+Bz1     # [T]

                    # Sys III -> CS
                    B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
                    x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
                    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
                    phi_cs = np.arctan2(y_cs, x_cs)
                    B_rho_cs = B_vec_cs[0] * \
                        np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
                    B_phi_cs = 0.0     # rho_cs - z_cs plane
                    B_z_cs = B_vec_cs[2]
                    B_vec_cs = np.array([
                        B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
                        B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
                        B_z_cs
                    ])

                    # CS -> Sys III
                    B_vec_s3 = RT @ B_vec_cs
                    B0 = math.sqrt(
                        B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

                    ds = ds0/(np.log10(B0*1E+9))**p      # [m]
                    s += ds

                    # 座標更新 (x, y, z)
                    x += (ds*B_vec_s3[0]/B0)*ns
                    y += (ds*B_vec_s3[1]/B0)*ns
                    z += (ds*B_vec_s3[2]/B0)*ns

                    # 座標更新 (r, theta, phi)
                    rs = math.sqrt(x**2 + y**2 + z**2)
                    theta = math.acos(z/rs)
                    phi = math.atan2(y, x)

                    # 終了条件 (北の外側境界)
                    if abs(z) > 55.0*RJ:
                        print('North edge [RJ]:')
                        i_foot_N[j, k] = np.nan
                        i_distance_N[j, k] = np.nan
                        break
                    # 終了条件 (水平距離の外側境界)
                    if math.sqrt(x**2+y**2) > 60.0*RJ:
                        print('Horizontal edge')
                        i_foot_N[j, k] = np.nan
                        i_distance_N[j, k] = np.nan
                        break
                    # 終了条件 (水平距離の外側境界)
                    if rs < (66854.0+500.0)*1E+3:
                        # CS座標での磁気余緯度 座標更新後に再計算
                        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
                        rho_cs = math.sqrt(x_cs**2 + y_cs**2)
                        theta_i = math.degrees(math.atan2(rho_cs, abs(z_cs)))
                        i_foot_N[j, k] = theta_i
                        i_distance_N[j, k] = s
                        break
                    # 終了していない
                    if i == Niter-1:
                        print('Tracing was not completed.')

    np.savetxt(
        'results/FAC_grid/ionos_colatitude_1224.txt',
        i_foot_N
    )

    np.savetxt(
        'results/FAC_grid/ionos_distance_1224.txt',
        i_distance_N
    )

In [ ]:
if grid_tracing:
    jm.Internal.Config(Model='jrm33', Degree=1, CartesianIn=True,
                       CartesianOut=True)
    jm.Con2020.Config(equation_type='integral')
    csfield = CSField()
    csfield.config(
        i_phi0=4.9213685925554586E-5,
        I_rho0=11.103883377723957,
        a0=3.46781481E-6,
        MLT0=0.52113394,
        a1=18.34895438,
        MLT1=20.0143857,
        b1=4.2518,
        MLTb=-0.2086,
    )
    # ==============================================================
    # FIELD LINE TRACE IN THE CURRENT SHEET PLANE (0600 MLT DAWN)
    # FROM THR MAGNETOSPHERE
    # ==============================================================
    rho_cs0 = RHO_CS_0060*RJ  # [m]
    phi_cs0 = 0.0             # [rad]
    z_cs0 = Z_CS_0060*RJ      # [m]

    x_cs0 = rho_cs0*np.cos(phi_cs0)     # [m]
    y_cs0 = rho_cs0*np.sin(phi_cs0)     # [m]

    R = csfield.R_sys3_2_cs()
    RT = csfield.R_cs_2_sys3()

    Niter = 50000

    ds0 = 500000.0   # [m]
    p = 1.2

    i_foot_N2 = np.zeros(x_cs0.shape)
    i_distance_N2 = np.zeros(x_cs0.shape)

    # 北半球電離圏に向かう向き
    # -1: north; 1: south
    ns = -1
    for k in range(x_cs0.shape[1]):
        for j in range(x_cs0.shape[0]):
            s = 0
            pos_s3_0 = RT @ np.array([
                x_cs0[j, k], y_cs0[j, k], z_cs0[j, k]
            ])
            x = pos_s3_0[0].copy()
            y = pos_s3_0[1].copy()
            z = pos_s3_0[2].copy()
            if z_cs0[j, k] >= 0.0*RJ:
                ns = -1
            elif z_cs0[j, k] < 0.0*RJ:
                ns = -1
                pos_s3_0 = RT @ np.array([
                    x_cs0[j, k], y_cs0[j, k], -z_cs0[j, k]
                ])
                x = pos_s3_0[0].copy()
                y = pos_s3_0[1].copy()
                z = pos_s3_0[2].copy()
            if np.isnan(J_para_0060[j, k]):
                i_foot_N2[j, k] = np.nan
                i_distance_N2[j, k] = np.nan
                continue
            else:
                for i in range(Niter):
                    Bx0, By0, Bz0 = jm.Internal.Field(
                        x/RJ, y/RJ, z/RJ)       # [nT]
                    Bx1, By1, Bz1 = csfield.magnetic_field_v2(
                        x/RJ, y/RJ, z/RJ,
                        MLT=6.0)  # [T]
                    Bx = (Bx0*1E-9)+Bx1     # [T]
                    By = (By0*1E-9)+By1     # [T]
                    Bz = (Bz0*1E-9)+Bz1     # [T]

                    # Sys III -> CS
                    B_vec_cs = R @ np.array([float(Bx), float(By), float(Bz)])
                    x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
                    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
                    phi_cs = np.arctan2(y_cs, x_cs)
                    B_rho_cs = B_vec_cs[0] * \
                        np.cos(phi_cs) + B_vec_cs[1]*np.sin(phi_cs)
                    B_phi_cs = 0.0     # rho_cs - z_cs plane
                    B_z_cs = B_vec_cs[2]
                    B_vec_cs = np.array([
                        B_rho_cs * np.cos(phi_cs) - B_phi_cs * np.sin(phi_cs),
                        B_rho_cs * np.sin(phi_cs) + B_phi_cs * np.cos(phi_cs),
                        B_z_cs
                    ])

                    # CS -> Sys III
                    B_vec_s3 = RT @ B_vec_cs
                    B0 = math.sqrt(
                        B_vec_s3[0]**2+B_vec_s3[1]**2+B_vec_s3[2]**2)      # [T]

                    ds = ds0/(np.log10(B0*1E+9))**p      # [m]
                    s += ds

                    # 座標更新 (x, y, z)
                    x += (ds*B_vec_s3[0]/B0)*ns
                    y += (ds*B_vec_s3[1]/B0)*ns
                    z += (ds*B_vec_s3[2]/B0)*ns

                    # 座標更新 (r, theta, phi)
                    rs = math.sqrt(x**2 + y**2 + z**2)
                    theta = math.acos(z/rs)
                    phi = math.atan2(y, x)

                    # 終了条件 (北の外側境界)
                    if abs(z) > 55.0*RJ:
                        print('North edge [RJ]:')
                        i_foot_N2[j, k] = np.nan
                        i_distance_N2[j, k] = np.nan
                        break
                    # 終了条件 (水平距離の外側境界)
                    if math.sqrt(x**2+y**2) > 60.0*RJ:
                        print('Horizontal edge')
                        i_foot_N2[j, k] = np.nan
                        i_distance_N2[j, k] = np.nan
                        break
                    # 終了条件 (水平距離の外側境界)
                    if rs < (66854.0+500.0)*1E+3:
                        # CS座標での磁気余緯度 座標更新後に再計算
                        x_cs, y_cs, z_cs = csfield._sys3_2_cs(x/RJ, y/RJ, z/RJ)
                        rho_cs = math.sqrt(x_cs**2 + y_cs**2)
                        theta_i = math.degrees(math.atan2(rho_cs, abs(z_cs)))
                        i_foot_N2[j, k] = theta_i
                        i_distance_N2[j, k] = s
                        break
                    # 終了していない
                    if i == Niter-1:
                        print('Tracing was not completed.')

    np.savetxt(
        'results/FAC_grid/ionos_colatitude_0060.txt',
        i_foot_N2
    )

    np.savetxt(
        'results/FAC_grid/ionos_distance_0060.txt',
        i_distance_N2
    )

In [ ]:
i_foot_N = np.loadtxt(
    'results/FAC_grid/ionos_colatitude_1224.txt'
)
i_distance_N = np.loadtxt(
    'results/FAC_grid/ionos_distance_1224.txt'
)

i_foot_N2 = np.loadtxt(
    'results/FAC_grid/ionos_colatitude_0060.txt'
)
i_distance_N2 = np.loadtxt(
    'results/FAC_grid/ionos_distance_0060.txt'
)

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 23
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=326)

# ===========================================================
# Common settings
# ===========================================================
for i in range(2):
    ax = axes[i]
    ax.set_facecolor('k')
    ax.set_xlim(0.0, 45.0)
    ax.set_ylim(-40.0, 15.0)
    ax.set_xticks(np.arange(0, 45+1, 5, dtype=int))
    ax.xaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_yticks(np.arange(-40, 15+1, 5, dtype=int))
    ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
    ax.set_xlabel(r'$\rho_{\rm cs}$ [$R_J$]')
    ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]')

    # ===========================================================
    # Jupiter
    # ===========================================================
    ax.add_patch(plt.Circle((0, 0), 1.0, color='tan',
                            zorder=6, fill=True, clip_on=True))

    # ===========================================================
    # Galilean moons
    # ===========================================================
    draw_Galilean_moons(ax)

    # ===========================================================
    # Hatch mask outside 36 RJ
    # ===========================================================
    hatch_outer_36rj(ax)


# ===========================================================
# Left panel (Dawn)
# ===========================================================
ax = axes[0]
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
)
ax.xaxis.set_inverted(True)
ax.text(0.05, 0.02,
        'Midnight-Dawn\n0-6 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='left',
        transform=ax.transAxes,
        zorder=10)

vmin = 0.0
vmax = 30.0
cmap = plt.get_cmap('turbo')
norm = mplcolors.Normalize(vmin=vmin, vmax=vmax)

cf = ax.imshow(
    i_foot_N2,
    norm=norm,
    cmap=cmap,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_0060.min(), RHO_CS_0060.max(),
        Z_CS_0060.min(),   Z_CS_0060.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)
# cs = ax.contour(RHO_CS_0060, Z_CS_0060, i_foot_N2, levels=20)
# ax.clabel(cs, inline=True, fontsize=15)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_0600(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dawn_midnight_latitude_label_left(ax)


# ===========================================================
# Right panel (Dusk)
# ===========================================================
ax = axes[1]
ax.set_ylabel(r'$z_{\rm cs}$ [$R_J$]',
              verticalalignment='bottom',
              rotation=270, labelpad=28,)
ax.tick_params(
    axis='both',       # Apply to both x and y axes
    which='both',      # Apply to both major and minor ticks
    top=True,          # Show top tick marks
    right=True,        # Show right tick marks
    labelleft=False,
    labelright=True,
)
ax.text(0.95, 0.02,
        'Duskside\n12-24 MLT',
        color=UC.lightbeige,
        fontsize=F.fontsize,
        verticalalignment='bottom',
        horizontalalignment='right',
        transform=ax.transAxes,
        zorder=10)
ax.yaxis.set_label_position('right')

vmin = 0.0
vmax = 30.0
cmap = plt.get_cmap('turbo')
norm = mplcolors.Normalize(vmin=vmin, vmax=vmax)

cf = ax.imshow(
    i_foot_N,
    norm=norm,
    cmap=cmap,
    origin='lower',           # z軸が下から上に増加する向きに合わせる
    extent=[
        RHO_CS_1224.min(), RHO_CS_1224.max(),
        Z_CS_1224.min(),   Z_CS_1224.max()
    ],
    aspect='auto',            # ax の縦横比に合わせて伸縮
    interpolation='nearest',  # pcolormesh と同じ「補間なし」の見た目
    zorder=1,
)
# cs = ax.contour(RHO_CS_1224, Z_CS_1224, i_foot_N, levels=20)
# ax.clabel(cs, inline=True, fontsize=15)

# ===========================================================
# Magnetic field line
# ===========================================================
draw_field_line_1800(ax)

# ===========================================================
# Latitude labels on arc of r=40 RJ
# ===========================================================
dusk_latitude_label_right(ax)


# ===========================================================
# Colorbar
# ===========================================================
fig.tight_layout()

axpos = ax.get_position()
cax = fig.add_axes([axpos.x1*1.1, axpos.y0, 0.02, axpos.y1-axpos.y0])
pp = fig.colorbar(cf, cax=cax)
pp.ax.set_yticks(np.arange(0, 30+1, 5))
pp.ax.yaxis.set_minor_locator(ptick.AutoMinorLocator(5))
pp.ax.set_ylabel(
    r'$\theta_{\rm i}$ [deg]',
    verticalalignment='bottom',
    rotation=270)

fig.suptitle(r'Ionospheric colatitude $\theta_{\rm i}$', y=1.01,
             fontsize=F.fontsize, fontweight="bold")

plt.savefig('img/FAC_v2/ionospheric_colatitude_grid.pdf',
            bbox_inches='tight')

In [ ]:
# θiビンごとにJ∥を平均
theta_bins = np.arange(2, 22, 0.5)  # [deg]

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060,  i_foot_N2, RHO_CS_0060, Z_CS_0060),
    ('12-18 MLT', J_para_1218,  i_foot_N, RHO_CS_1218, Z_CS_1218),
    ('18-24 MLT', J_para_1824,  i_foot_N, RHO_CS_1824, Z_CS_1824),
]:
    profile = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        profile.append(np.nanmean(J_grid[mask]) if mask.sum() > 0 else np.nan)
    plt.plot(theta_bins[:-1]+0.25, profile, label=MLT_label)

plt.axhline(0, color='gray', lw=0.5)
plt.xlabel(r'$\theta_i$ [deg]')
plt.ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]')
plt.legend()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.pcolormesh(RHO_CS_0060, Z_CS_0060, i_distance_N2/RJ,
               cmap='viridis', shading='auto')
plt.colorbar(label='Distance to ionosphere [RJ]')
plt.title('Dawn\nDistance to ionosphere')

plt.subplot(1, 2, 2)
plt.pcolormesh(RHO_CS_1224, Z_CS_1224, i_distance_N/RJ,
               cmap='viridis', shading='auto')
plt.colorbar(label='Distance to ionosphere [RJ]')
plt.title('Dusk\nDistance to ionosphere')

In [ ]:
# ===========================================================
# プロット (5-36 RJ)
# ===========================================================
F = ShareXaxis()
F.fontsize = 21
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.8, 4.), dpi='XL')
F.initialize()

F.set_xaxis(label=r'Ionospheric colatitude $\theta_i$ [deg]',
            min=0,
            max=24,
            ticks=np.arange(0, 24+1, 4),
            ticklabels=np.arange(0, 24+1, 4),
            minor_num=4)
F.set_yaxis(ax_idx=0,
            label=r'Inward $J_\parallel$ [nA m$^{-2}$]',
            min=-0.03, max=0.03,
            ticks=np.linspace(-3, 3, 7)/100,
            ticklabels=np.linspace(-3, 3, 7)/100,
            minor_num=5)

theta_bins = np.arange(2, 22, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}
shades = {'0-6 MLT': UC.pink, '12-24 MLT': UC.lightblue}
markers = {'0-6 MLT': '^', '12-24 MLT': 'v'}

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, RHO_CS_0060, Z_CS_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, RHO_CS_1224, Z_CS_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]
    marker = markers[MLT_label]
    shade = shades[MLT_label]

    # 中央値
    F.ax.plot(theta_centers, profile_median,
              marker=marker, markersize=4,
              markeredgecolor='k', markerfacecolor='w',
              markeredgewidth=0.7,
              color=color, linewidth=1.7, label=MLT_label)

    # 四分位範囲
    F.ax.fill_between(theta_centers, profile_q25, profile_q75,
                      color=shade, alpha=0.4)


F.ax.set_title('Inward FAC density', weight='bold')

F.ax.grid(color=UC.lightgray, linewidth=0.35, zorder=0.1)
F.ax.axhline(y=0, color='k', linewidth=0.5, zorder=0.1)

legend = F.legend(ax_idx=0,
                  ncol=2, markerscale=1.25,
                  loc='lower right',
                  handlelength=0.72,
                  textcolor=True,
                  fontsize_scale=0.8,
                  handletextpad=0.4)
legend_shadow(legend=legend, fig=F.fig, ax=F.ax, d=0.7)

plt.savefig('img/FAC_v2/FAC_vs_ionospheric_colatitude_36rj.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット (5-36 RJ)
# ===========================================================
F = ShareXaxis()
F.fontsize = 21
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.8, 4.), dpi='XL')
F.initialize()

F.set_xaxis(label=r'Ionospheric colatitude $\theta_i$ [deg]',
            min=0,
            max=24,
            ticks=np.arange(0, 24+1, 4),
            ticklabels=np.arange(0, 24+1, 4),
            minor_num=4)
F.set_yaxis(ax_idx=0,
            label=r'Inward $J_\parallel$ [nA m$^{-2}$]',
            min=-0.03, max=0.03,
            ticks=np.linspace(-3, 3, 7)/100,
            ticklabels=np.linspace(-3, 3, 7)/100,
            minor_num=5)

theta_bins = np.arange(2, 22, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}
shades = {'0-6 MLT': UC.pink, '12-24 MLT': UC.lightblue}
markers = {'0-6 MLT': '^', '12-24 MLT': 'v'}

unreliable = -np.sign(Z_CS_1224)*J_para_1224\
    - (-np.sign(Z_CS_1224_0)*J_para_1224_0)

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, RHO_CS_0060, Z_CS_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, RHO_CS_1224, Z_CS_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0) * (abs(unreliable) < 0.0025)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]
    marker = markers[MLT_label]
    shade = shades[MLT_label]

    # 中央値
    F.ax.plot(theta_centers, profile_median,
              marker=marker, markersize=4,
              markeredgecolor='k', markerfacecolor='w',
              markeredgewidth=0.7,
              color=color, linewidth=1.7, label=MLT_label)

    # 四分位範囲
    F.ax.fill_between(theta_centers, profile_q25, profile_q75,
                      color=shade, alpha=0.4)


F.ax.set_title('Inward FAC density (masked)', weight='bold')

F.ax.grid(color=UC.lightgray, linewidth=0.35, zorder=0.1)
F.ax.axhline(y=0, color='k', linewidth=0.5, zorder=0.1)

legend = F.legend(ax_idx=0,
                  ncol=2, markerscale=1.25,
                  loc='lower right',
                  handlelength=0.72,
                  textcolor=True,
                  fontsize_scale=0.8,
                  handletextpad=0.4)
legend_shadow(legend=legend, fig=F.fig, ax=F.ax, d=0.7)

# plt.savefig('img/FAC_v2/FAC_vs_ionospheric_colatitude_36rj.pdf',
#             bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット (5-25 RJ)
# ===========================================================
F = ShareXaxis()
F.fontsize = 21
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.8, 4.), dpi='XL')
F.initialize()

F.set_xaxis(label=r'Ionospheric colatitude $\theta_i$ [deg]',
            min=0,
            max=24,
            ticks=np.arange(0, 24+1, 4),
            ticklabels=np.arange(0, 24+1, 4),
            minor_num=4)
F.set_yaxis(ax_idx=0,
            label=r'Inward $J_\parallel$ [nA m$^{-2}$]',
            min=-0.03, max=0.03,
            ticks=np.linspace(-3, 3, 7)/100,
            ticklabels=np.linspace(-3, 3, 7)/100,
            minor_num=5)

theta_bins = np.arange(2, 22, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}
shades = {'0-6 MLT': UC.pink, '12-24 MLT': UC.lightblue}
markers = {'0-6 MLT': '^', '12-24 MLT': 'v'}

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, RHO_CS_0060, Z_CS_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, RHO_CS_1224, Z_CS_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 25.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]
    marker = markers[MLT_label]
    shade = shades[MLT_label]

    # 中央値
    F.ax.plot(theta_centers, profile_median,
              marker=marker, markersize=4,
              markeredgecolor='k', markerfacecolor='w',
              markeredgewidth=0.7,
              color=color, linewidth=1.7, label=MLT_label)

    # 四分位範囲
    F.ax.fill_between(theta_centers, profile_q25, profile_q75,
                      color=shade, alpha=0.4)


F.ax.set_title('Inward FAC density', weight='bold')

F.ax.grid(color=UC.lightgray, linewidth=0.35, zorder=0.1)
F.ax.axhline(y=0, color='k', linewidth=0.5, zorder=0.1)

F.ax.text(0.025, 0.94,
          r'$r \leq 25$ $R_{\rm J}$',
          color='k',
          fontsize=F.fontsize*0.82,
          verticalalignment='top',
          horizontalalignment='left',
          transform=F.ax.transAxes)

legend = F.legend(ax_idx=0,
                  ncol=1, markerscale=1.25,
                  loc='upper right',
                  handlelength=0.72,
                  textcolor=True,
                  fontsize_scale=0.75,
                  handletextpad=0.4)
legend_shadow(legend=legend, fig=F.fig, ax=F.ax, d=0.7)

plt.savefig('img/FAC_v2/FAC_vs_ionospheric_colatitude_25rj.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット (5-36 RJ)
# ===========================================================
F = ShareXaxis()
F.fontsize = 21
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.8, 4.), dpi='XL')
F.initialize()

F.set_xaxis(label=r'Ionospheric colatitude $\theta_i$ [deg]',
            min=0,
            max=24,
            ticks=np.arange(0, 24+1, 4),
            ticklabels=np.arange(0, 24+1, 4),
            minor_num=4)
F.set_yaxis(ax_idx=0,
            label=r'Inward $J_\parallel$ [nA m$^{-2}$]',
            min=-0.03, max=0.03,
            ticks=np.linspace(-3, 3, 7)/100,
            ticklabels=np.linspace(-3, 3, 7)/100,
            minor_num=5)

theta_bins = np.arange(2, 22, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red,
          '12-18 MLT': UC.blue,
          '18-24 MLT': UC.green}
shades = {'0-6 MLT': UC.pink,
          '12-18 MLT': UC.lighterblue,
          '18-24 MLT': UC.lightgreen}
markers = {'0-6 MLT': '^',
           '12-18 MLT': 'v',
           '18-24 MLT': '.'}

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, RHO_CS_0060, Z_CS_0060),
    ('12-18 MLT', J_para_1218, i_foot_N, RHO_CS_1218, Z_CS_1218),
    ('18-24 MLT', J_para_1824, i_foot_N, RHO_CS_1824, Z_CS_1824),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]
    marker = markers[MLT_label]
    shade = shades[MLT_label]

    # 中央値
    F.ax.plot(theta_centers, profile_median,
              marker=marker, markersize=4,
              markeredgecolor='k', markerfacecolor='w',
              markeredgewidth=0.7,
              color=color, linewidth=1.7, label=MLT_label)

    # 四分位範囲
    F.ax.fill_between(theta_centers, profile_q25, profile_q75,
                      color=shade, alpha=0.6)


F.ax.set_title('Inward FAC density', weight='bold')

F.ax.grid(color=UC.lightgray, linewidth=0.35, zorder=0.1)
F.ax.axhline(y=0, color='k', linewidth=0.5, zorder=0.1)

legend = F.legend(ax_idx=0,
                  ncol=3, markerscale=1.25,
                  loc='upper center',
                  handlelength=0.72,
                  textcolor=True,
                  fontsize_scale=0.75,
                  handletextpad=0.4)
legend_shadow(legend=legend, fig=F.fig, ax=F.ax, d=0.7)

plt.savefig('img/FAC_v2/FAC_vs_ionospheric_colatitude_3_sector.pdf',
            bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 19
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, ax = plt.subplots(2, 1, figsize=(6, 7), dpi=326)

theta_bins = np.arange(2, 24, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}

i = 0

for MLT_label, J_grid, theta_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, Z_CS_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, Z_CS_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]

    ax[i].set_xlabel(r'Ionospheric colatitude $\theta_{\rm i}$ [deg]')
    ax[i].set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]')
    ax[i].set_xlim(0.0, 30.0)
    ax[i].set_ylim(-0.05, 0.05)
    ax[i].set_xticks(np.arange(0, 30+1, 5))
    ax[i].set_yticks(np.arange(-5, 5+0.1, 2.5)/100)

    ax2 = ax[i].twinx()
    ax2.bar(theta_centers, profile_n,
            width=0.4, alpha=0.2, color=UC.gray,
            label='N points')
    ax2.set_ylabel('N data points', color=UC.gray)

    i += 1
ax[0].scatter(i_foot_N2, J_para_0060, c=UC.red, s=6., lw=0, alpha=0.18)
ax[1].scatter(i_foot_N, J_para_1224, c=UC.blue, s=6., lw=0, alpha=0.18)

ax[0].set_title('Local inward FAC density', fontweight='bold')

fig.tight_layout()

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 19
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, ax = plt.subplots(2, 1, figsize=(6, 7), dpi=326)

theta_bins = np.arange(2, 24, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}

i = 0

for MLT_label, J_grid, theta_grid, z_cs_grid, B0_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, Z_CS_0060, B0_grid_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, Z_CS_1224, B0_grid_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    # JRM33ダイポール（n=1）での電離層磁場強度
    # B_dipole(r, θ) = (B0/r³) * sqrt(1 + 3cos²θ)
    # B0 = g10 = 410248 nT at r = 1 RJ
    g10 = 410248.0E-9  # [T] JRM33 dipole
    R_iono = (66854.0+500.0)/71492.0  # [RJ]

    # θi [deg] → B_iono [nT]
    theta_i_rad = np.radians(theta_grid)  # [rad]
    B_iono = (g10/R_iono**3)*np.sqrt(1+3*np.cos(theta_i_rad)**2)

    # 規格化
    J_para_iono = J_grid*(B0_grid/B_iono)*1e9  # [nA/m²]

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid_masked = -np.sign(z_cs_grid[mask0])*J_para_iono[mask0]
    # J_grid_masked = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    print(np.nanmax(J_grid_masked))

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (
            theta_grid < t1) & ~np.isnan(J_grid_masked)
        vals = J_grid_masked[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]

    ax[i].set_xlabel(r'Ionospheric colatitude $\theta_{\rm i}$ [deg]')
    ax[i].set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]')
    # ax[i].set_xlim(0.0, 30.0)
    ax[i].set_ylim(-1500, 1500)
    # ax[i].set_xticks(np.arange(0, 30+1, 5))
    # ax[i].set_yticks(np.arange(-5, 5+0.1, 2.5)/100)

    ax2 = ax[i].twinx()
    ax2.bar(theta_centers, profile_n,
            width=0.4, alpha=0.2, color=UC.gray,
            label='N points')
    ax2.set_ylabel('N data points', color=UC.gray)

    ax[i].scatter(theta_grid, J_grid_masked, c=UC.red, s=6., lw=0, alpha=0.18)

    i += 1

ax[0].set_title('Inward FAC density on ionosphere', fontweight='bold')

fig.tight_layout()

In [ ]:
# ===========================================================
# プロット (5-36 RJ)
# ===========================================================
F = ShareXaxis()
F.fontsize = 21
F.fontname = 'Liberation Sans Narrow'

F.set_figparams(nrows=1, figsize=(5.8, 4.), dpi='XL')
F.initialize()

F.set_xaxis(label=r'Ionospheric colatitude $\theta_i$ [deg]',
            min=0,
            max=24,
            ticks=np.arange(0, 24+1, 4),
            ticklabels=np.arange(0, 24+1, 4),
            minor_num=4)
F.set_yaxis(ax_idx=0,
            label=r'Inward $J_\parallel$ [nA m$^{-2}$]',
            min=-2000, max=2000,
            ticks=np.linspace(-20, 20, 5)*100,
            ticklabels=np.linspace(-20, 20, 5)*100,
            minor_num=5)

theta_bins = np.arange(2, 22, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}
shades = {'0-6 MLT': UC.pink, '12-24 MLT': UC.lightblue}
markers = {'0-6 MLT': '^', '12-24 MLT': 'v'}

for MLT_label, J_grid, theta_grid, rho_cs_grid, z_cs_grid, B0_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, RHO_CS_0060, Z_CS_0060, B0_grid_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, RHO_CS_1224, Z_CS_1224, B0_grid_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    # JRM33ダイポール（n=1）での電離層磁場強度
    # B_dipole(r, θ) = (B0/r³) * sqrt(1 + 3cos²θ)
    # B0 = g10 = 410248 nT at r = 1 RJ
    g10 = 410248.0E-9  # [T] JRM33 dipole
    R_iono = (66854.0+500.0)/71492.0  # [RJ]

    # θi [deg] → B_iono [nT]
    theta_i_rad = np.radians(theta_grid)  # [rad]
    B_iono = (g10/R_iono**3)*np.sqrt(1+3*np.cos(theta_i_rad)**2)

    # 規格化
    J_para_iono = J_grid*(B0_grid/B_iono)*1e9  # [nA/m²]

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    # J_grid_masked = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    J_grid_masked = -np.sign(z_cs_grid[mask0])*J_para_iono[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (
            theta_grid < t1) & ~np.isnan(J_grid_masked)
        vals = J_grid_masked[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]
    marker = markers[MLT_label]
    shade = shades[MLT_label]

    # 中央値
    F.ax.plot(theta_centers, profile_median,
              marker=marker, markersize=4,
              markeredgecolor='k', markerfacecolor='w',
              markeredgewidth=0.7,
              color=color, linewidth=1.7, label=MLT_label)

    # 四分位範囲
    F.ax.fill_between(theta_centers, profile_q25, profile_q75,
                      color=shade, alpha=0.4)


F.ax.set_title('Inward FAC density on ionosphere', weight='bold')

F.ax.grid(color=UC.lightgray, linewidth=0.35, zorder=0.1)
F.ax.axhline(y=0, color='k', linewidth=0.5, zorder=0.1)

legend = F.legend(ax_idx=0,
                  ncol=2, markerscale=1.25,
                  loc='lower right',
                  handlelength=0.72,
                  textcolor=True,
                  fontsize_scale=0.8,
                  handletextpad=0.4)
legend_shadow(legend=legend, fig=F.fig, ax=F.ax, d=0.7)

# plt.savefig('img/FAC_v2/FAC_vs_ionospheric_colatitude_36rj.pdf',
#             bbox_inches='tight')

In [ ]:
# ===========================================================
# プロット
# ===========================================================
F = ShareXaxis()
F.fontsize = 19
F.fontname = 'Liberation Sans Narrow'
F.set_default()

fig, ax = plt.subplots(2, 1, figsize=(6, 7), dpi=326)

theta_bins = np.arange(2, 24, 0.5)  # [deg]
theta_centers = theta_bins[:-1] + 0.25

colors = {'0-6 MLT': UC.red, '12-24 MLT': UC.blue}

i = 0

for MLT_label, J_grid, theta_grid, z_cs_grid in [
    ('0-6 MLT',   J_para_0060, i_foot_N2, Z_CS_0060),
    ('12-24 MLT', J_para_1224, i_foot_N, Z_CS_1224),
]:
    profile_median = []
    profile_q25 = []
    profile_q75 = []
    profile_n = []

    mask0 = (rho_cs_grid >= 5.0) & (
        np.sqrt(rho_cs_grid**2+z_cs_grid**2) <= 36.0)
    J_grid = -np.sign(z_cs_grid[mask0])*J_grid[mask0]
    theta_grid = theta_grid[mask0]

    for t0, t1 in zip(theta_bins[:-1], theta_bins[1:]):
        mask = (theta_grid >= t0) & (theta_grid < t1) & ~np.isnan(J_grid)
        vals = J_grid[mask]
        n = mask.sum()
        profile_n.append(n)

        if n >= 5:  # データ点が5点以上のビンのみ
            profile_median.append(np.nanmedian(vals))
            profile_q25.append(np.nanpercentile(vals, 25))
            profile_q75.append(np.nanpercentile(vals, 75))
        else:
            profile_median.append(np.nan)
            profile_q25.append(np.nan)
            profile_q75.append(np.nan)

    profile_median = np.array(profile_median)
    profile_q25 = np.array(profile_q25)
    profile_q75 = np.array(profile_q75)
    color = colors[MLT_label]

    ax[i].set_xlabel(r'Ionospheric colatitude $\theta_{\rm i}$ [deg]')
    ax[i].set_ylabel(r'Inward $J_\parallel$ [nA m$^{-2}$]')
    ax[i].set_xlim(0.0, 30.0)
    ax[i].set_ylim(-0.01, 0.01)
    ax[i].set_xticks(np.arange(0, 30+1, 5))
    ax[i].set_yticks(np.arange(-1, 1+0.1, 0.5)/100)

    ax2 = ax[i].twinx()
    ax2.bar(theta_centers, profile_n,
            width=0.4, alpha=0.2, color=UC.gray,
            label='N points')
    ax2.set_ylabel('N data points', color=UC.gray)

    i += 1

vals = -np.sign(Z_CS_1224)*J_para_1224\
    - (-np.sign(Z_CS_1224_0)*J_para_1224_0)

ax[1].scatter(i_foot_N, vals, c=UC.blue, s=6., lw=0, alpha=0.18)

ax[0].set_title('Local inward FAC density difference', fontweight='bold')

fig.tight_layout()